# V3 pipeline - RESUME notebook (Conditional DDPM -> end of Phase 5)

Re-runs the **full** pipeline on a fresh Colab runtime using the checkpoints already on Drive, but **fast-forwards past work that was already completed** so you don't pay for it twice:

- Training (v2/v3 classifiers, restorers) loads from checkpoints via the existing resume-guards.
- **Stubbed** (already done, slow): Optuna HPO, knowledge distillation, and the Phase-3 SHAP/IG XAI benchmark + aggregation. The XAI results are reloaded from `xai_results.csv`.
- **CycleGAN is disabled** (removed from the enhancer list and build step).
- Everything from **Conditional DDPM through Phase 5** runs fresh and produces results.

**Usage:** GPU runtime -> Run all. First run still pays one-time data extraction + pip install.


# V3 PATCH MARKER (do not delete)

V3-extended notebook built by `build_v3.py` from version2. Adds:

- Step 7: pathology-preserving conditional DDPM (Phase 4 Diffusion #3) — classifier-feature perceptual loss + input-fidelity loss + sampling-time anti-hallucination clamp. New checkpoint `ddpm_pathology_v1.pt`, enhancer key `'ddpm_path'`.
- Step 8: hallucination detection report (grade-change / pixel-deviation / risk), saved to `phase4/metrics/ddpm_pathology_hallucination.csv`.

The vanilla DDPM (Step 4) is retained so the recovery table reports both DDPMs (vanilla vs pathology) as an ablation. All new training is resume-guarded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm captum shap grad-cam scikit-image opencv-python-headless tabulate
!pip install -q diffusers transformers accelerate

In [ ]:
from pathlib import Path
import zipfile

# ----- Drive (input + persistent output) -----
DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
THESIS_ZIP      = DRIVE_ROOT / 'Thesis.zip'          # single outer zip
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'

In [ ]:
LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
EYEQ_DIR     = LOCAL_ROOT / 'eyeq'
PRISTINE_DIR = LOCAL_ROOT / 'pristine'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
ENHANCED_DIR = LOCAL_ROOT / 'enhanced'

PHASE_DIRS = {
    'phase1_data_engineering':   RESULTS_ROOT / 'phase1_data_engineering',
    'phase2_model_benchmarking': RESULTS_ROOT / 'phase2_model_benchmarking',
    'phase3_xai_benchmark':      RESULTS_ROOT / 'phase3_xai_benchmark',
    'phase4_genai_enhancement':  RESULTS_ROOT / 'phase4_genai_enhancement',
    'phase5_quality_ensemble':   RESULTS_ROOT / 'phase5_quality_ensemble',
}
PHASE_SUBDIRS = ('metrics', 'plots', 'samples', 'logs')

In [ ]:
for p in [LOCAL_ROOT, PRISTINE_DIR, DEGRADED_DIR, ENHANCED_DIR,
          RESULTS_ROOT, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for phase_dir in PHASE_DIRS.values():
    for sub in PHASE_SUBDIRS:
        (phase_dir / sub).mkdir(parents=True, exist_ok=True)

print('Drive zip   :', THESIS_ZIP, '(exists:', THESIS_ZIP.exists(), ')')
print('Drive results tree:')
for k, v in PHASE_DIRS.items():
    print(f'  {k:35s} -> {v}')

In [ ]:
# Idempotent extraction — skips automatically if already done in this Colab session.
# Saves 5-15 minutes per reconnect. Force re-extract: smart_extract(force=True).
import zipfile, shutil, time

EXTRACT_MARK = LOCAL_ROOT / ".extracted_ok"        # sentinel file


def _has_data():
    if not EXTRACT_MARK.exists():
        return False
    has_aptos = any(APTOS_DIR.rglob("*.png")) or any(APTOS_DIR.rglob("*.jpg"))
    has_eyeq  = any(EYEQ_DIR.rglob("*.png"))  or any(EYEQ_DIR.rglob("*.jpg"))
    return has_aptos and has_eyeq


def _find_dir(root: Path, keywords):
    for p in root.rglob("*"):
        if p.is_dir() and any(k in p.name.lower() for k in keywords):
            return p
    return None


def smart_extract(force: bool = False):
    """Extract Thesis.zip -> /content/data, then recursively unpack inner zips.
    Skips if a previous extraction is still in place."""
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    if _has_data() and not force:
        n_aptos = sum(1 for _ in APTOS_DIR.rglob("*.*"))
        n_eyeq  = sum(1 for _ in EYEQ_DIR.rglob("*.*"))
        print(f"[skip] data already extracted (aptos={n_aptos}, eyeq={n_eyeq})")
        return

    if not THESIS_ZIP.exists():
        raise FileNotFoundError(f"Missing zip: {THESIS_ZIP}")

    t0 = time.time()
    print(f"[extract] {THESIS_ZIP.name} -> {LOCAL_ROOT}")
    with zipfile.ZipFile(THESIS_ZIP) as z:
        z.extractall(LOCAL_ROOT)

    while True:
        inners = list(LOCAL_ROOT.rglob("*.zip"))
        if not inners:
            break
        for iz in inners:
            target = iz.parent / iz.stem
            target.mkdir(exist_ok=True)
            print(f"  inner zip: {iz.name} -> {target}")
            with zipfile.ZipFile(iz) as z:
                z.extractall(target)
            iz.unlink()

    # Symlink canonical paths
    aptos_src = _find_dir(LOCAL_ROOT, ["aptos"])
    eyeq_src  = _find_dir(LOCAL_ROOT, ["eyeq", "eye_q", "eye-q"])
    print("Detected APTOS dir:", aptos_src)
    print("Detected EyeQ  dir:", eyeq_src)
    for canonical, real in [(APTOS_DIR, aptos_src), (EYEQ_DIR, eyeq_src)]:
        if real is None:
            print(f"[warn] {canonical.name} not found — check Thesis.zip contents")
            continue
        if canonical.exists() and canonical.resolve() != real.resolve():
            if canonical.is_dir() and not canonical.is_symlink():
                shutil.rmtree(canonical)
            else:
                canonical.unlink()
        if not canonical.exists():
            canonical.symlink_to(real, target_is_directory=True)
        print(f"  {canonical} -> {real}")

    EXTRACT_MARK.touch()
    print(f"[extract] done in {time.time()-t0:.1f}s")


smart_extract()         # subsequent calls in this session are instant


In [ ]:
def show_tree(root: Path, max_depth: int = 3):
    root = Path(root)
    for p in sorted(root.rglob('*')):
        depth = len(p.relative_to(root).parts)
        if depth > max_depth: continue
        kind = '/' if p.is_dir() else ''
        print(f'{"  "*(depth-1)}{p.name}{kind}')

print('=== APTOS ==='); show_tree(APTOS_DIR, max_depth=2)
print('\n=== EYEQ  ==='); show_tree(EYEQ_DIR,  max_depth=2)

In [ ]:
import torch
PHASE_DIRS = {
    'phase1_data_engineering':   RESULTS_ROOT / 'phase1_data_engineering',
    'phase2_model_benchmarking': RESULTS_ROOT / 'phase2_model_benchmarking',
    'phase3_xai_benchmark':      RESULTS_ROOT / 'phase3_xai_benchmark',
    'phase4_genai_enhancement':  RESULTS_ROOT / 'phase4_genai_enhancement',
    'phase5_quality_ensemble':   RESULTS_ROOT / 'phase5_quality_ensemble',
}
for p in [DEGRADED_DIR, ENHANCED_DIR, RESULTS_ROOT, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for d in PHASE_DIRS.values():
    for sub in ('metrics', 'plots', 'samples', 'logs'):
        (d / sub).mkdir(parents=True, exist_ok=True)

# Auto-detect APTOS train CSV + image dir (case-insensitive)
APTOS_CSV    = next(p for p in APTOS_DIR.rglob('*.csv') if 'train' in p.name.lower())
APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('*train_images*') if p.is_dir())
EYEQ_CSV     = next(EYEQ_DIR.rglob('*.csv'), None)

# Common constants
NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
IMAGE_SIZE  = 224
SEED        = 42
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15

MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50',
               'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}
TRAIN_CFG = dict(batch_size=32, num_workers=2, epochs=15,
                 lr=3e-4, weight_decay=1e-4, label_smoothing=0.05,
                 mixed_precision=True)

DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')
DEGRADATION_PARAMS = {
    'blur':     {'low': 2.0,  'mid': 5.0,  'high': 9.0},
    'exposure': {'low': 0.7,  'mid': 0.4,  'high': 0.2},
    'noise':    {'low': 0.02, 'mid': 0.06, 'high': 0.12},
}
EYEQ_QUALITY_LABELS = {0: 'good', 1: 'usable', 2: 'reject'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('APTOS csv   :', APTOS_CSV)
print('APTOS images:', APTOS_IMAGES)
print('EyeQ csv    :', EYEQ_CSV)
print('Device      :', DEVICE)

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms as T
import timm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from scipy.stats import spearmanr
from skimage.segmentation import slic

TFM_TRAIN = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.RandomHorizontalFlip(),
                        T.RandomRotation(15),
                        T.ColorJitter(0.1, 0.1, 0.1),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
TFM_EVAL  = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def load_tensor(path: Path) -> torch.Tensor:
    return TFM_EVAL(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)

In [ ]:
class APTOSDataset(Dataset):
    """Resolves images via IMAGE_INDEX so any extension / nested layout works."""
    def __init__(self, csv_path, images_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir)   # kept for API compatibility, not used
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

class FolderDataset(Dataset):
    """Reads `<root>/manifest.csv` (cols: id_code, diagnosis, rel_path)."""
    def __init__(self, root, transform=TFM_EVAL):
        self.root = Path(root); self.df = pd.read_csv(self.root / 'manifest.csv')
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['rel_path']).convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

In [ ]:
# === Improved training infrastructure ===
import math
from contextlib import contextmanager
from collections import Counter
from torch.utils.data import WeightedRandomSampler, Subset
from sklearn.model_selection import train_test_split

# 1) STRATIFIED SPLIT — keeps class proportions consistent across train/val/test
def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx = np.arange(len(labels)); labels = np.array(labels)
    tr_idx, te_idx = train_test_split(idx, test_size=test_frac, stratify=labels, random_state=seed)
    rel_val = val_frac / (1 - test_frac)
    tr_idx, va_idx = train_test_split(tr_idx, test_size=rel_val,
                                       stratify=labels[tr_idx], random_state=seed)
    return tr_idx.tolist(), va_idx.tolist(), te_idx.tolist()

# 2) CLASS WEIGHTS + BALANCED SAMPLER
def class_weights_tensor(labels, num_classes=NUM_CLASSES):
    cnt = Counter(labels); n = sum(cnt.values())
    return torch.tensor([n / (num_classes * max(cnt.get(i, 0), 1)) for i in range(num_classes)],
                         dtype=torch.float)

def balanced_sampler(labels):
    cnt = Counter(labels)
    w = [1.0 / cnt[l] for l in labels]
    return WeightedRandomSampler(w, num_samples=len(w), replacement=True)

# 3) FOCAL LOSS (handles soft targets from MixUp + per-class weights)
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.alpha, self.ls = gamma, alpha, label_smoothing
    def forward(self, logits, targets):
        if targets.dim() > 1:                       # soft labels (MixUp)
            log_p = F.log_softmax(logits, dim=-1)
            return -(targets * log_p).sum(-1).mean()
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                              reduction='none', label_smoothing=self.ls)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

# 4) EMA — exponential moving average of weights, +0.5–1% reliably
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
    def update(self, model):
        with torch.no_grad():
            for n, p in model.named_parameters():
                if p.requires_grad:
                    self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    @contextmanager
    def applied(self, model):
        backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.shadow[n])
        try: yield
        finally:
            for n, p in model.named_parameters():
                if p.requires_grad: p.data.copy_(backup[n])

# 5) STRONG TRAIN AUGMENTATION (medical-imaging safe — no extreme hue shifts)
TFM_TRAIN_STRONG = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(25),
    T.RandAugment(num_ops=2, magnitude=7),
    T.ColorJitter(0.15, 0.15, 0.15, 0.03),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.25),
])

# 6) MIXUP / CUTMIX (timm's implementation)
from timm.data import Mixup

# 7) LAYER-WISE LR DECAY (LRD) — critical for ViT fine-tuning
def vit_lrd_param_groups(model, base_lr, decay=0.75, weight_decay=5e-2):
    n_layers = len(model.blocks) + 1
    groups = []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith('head'):                                  depth = n_layers
        elif n.startswith('blocks'):                              depth = int(n.split('.')[1])
        elif any(n.startswith(s) for s in
                 ('patch_embed', 'cls_token', 'pos_embed', 'norm')): depth = 0
        else:                                                     depth = 0
        lr_scale = decay ** (n_layers - depth)
        groups.append({'params': p, 'lr': base_lr * lr_scale,
                       'weight_decay': 0.0 if p.ndim <= 1 else weight_decay})
    return groups

# 8) WARMUP + COSINE LR SCHEDULE
def warmup_cosine(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    def lr_lambda(step):
        epoch = step / max(steps_per_epoch, 1)
        if epoch < warmup_epochs:
            return epoch / max(warmup_epochs, 1)
        prog = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# 9) PER-ARCHITECTURE HYPERPARAMETERS
HPARAMS = {
    'resnet50':        dict(lr=3e-4, wd=1e-4, epochs=25, mixup=0.2, ls=0.1, gamma=1.5, warmup=3, llrd=None),
    'efficientnet_b3': dict(lr=2e-4, wd=1e-5, epochs=25, mixup=0.2, ls=0.1, gamma=2.0, warmup=3, llrd=None),
    'vit_base':        dict(lr=5e-5, wd=5e-2, epochs=30, mixup=0.4, ls=0.1, gamma=2.0, warmup=5, llrd=0.75),
}

print('Improved training infrastructure loaded.')

In [ ]:
def find_in_folder(folder, id_code, exts=('.jpg', '.png', '.jpeg')):
    """Return the actual file path for id_code in folder, regardless of extension."""
    for ext in exts:
        p = Path(folder) / f'{id_code}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'{id_code} not found in {folder} (tried {exts})')

In [ ]:
def build_model(name, pretrained=True):
    m = timm.create_model(TIMM_NAMES[name], pretrained=pretrained, num_classes=NUM_CLASSES)
    m._dr_arch = name
    return m

def load_classifier(name):
    m = build_model(name, pretrained=False).to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE, weights_only=False)
    m.load_state_dict(state['state_dict']); m.eval()
    return m

def get_xai_target_layer(model):
    a = model._dr_arch
    if a == 'resnet50':        return model.layer4[-1]
    if a == 'efficientnet_b3': return model.blocks[-1]
    if a == 'vit_base':        return model.blocks[-1].norm1
    raise ValueError(a)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, probs = [], [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        prob = torch.softmax(model(x), 1).cpu().numpy()
        probs.append(prob); ps.append(prob.argmax(1)); ys.append(y.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps); pr = np.concatenate(probs)
    out = {'accuracy': accuracy_score(y, p), 'f1_macro': f1_score(y, p, average='macro')}
    try: out['auc_macro_ovr'] = roc_auc_score(y, pr, average='macro', multi_class='ovr')
    except ValueError: out['auc_macro_ovr'] = float('nan')
    return out

def train_one(model, train_loader, val_loader, ckpt_path, history_csv,
              epochs=TRAIN_CFG['epochs'], lr=TRAIN_CFG['lr'],
              wd=TRAIN_CFG['weight_decay'], ls=TRAIN_CFG['label_smoothing'],
              amp=TRAIN_CFG['mixed_precision']):
    model = model.to(DEVICE)
    optim  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    crit   = nn.CrossEntropyLoss(label_smoothing=ls)
    scaler = GradScaler(enabled=amp)
    best_f1, history = -1.0, []
    for ep in range(epochs):
        model.train(); running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f'epoch {ep+1}/{epochs}', leave=False)
        for x, y, _ in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            with autocast(enabled=amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
            running += loss.item() * x.size(0); seen += x.size(0)
            pbar.set_postfix(loss=running/seen)
        sched.step()
        m = evaluate(model, val_loader)
        history.append({'epoch': ep+1, 'train_loss': running/seen, **m})
        print(f'  val: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}  auc={m["auc_macro_ovr"]:.4f}')
        if m['f1_macro'] > best_f1:
            best_f1 = m['f1_macro']
            torch.save({'state_dict': model.state_dict(), 'arch': model._dr_arch, 'val': m}, ckpt_path)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    return pd.DataFrame(history)

In [ ]:
def gradcam_heatmap(model, x, target_class=None):
    layer = get_xai_target_layer(model)
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda _, __, o: acts.update(v=o.detach()))
    h2 = layer.register_full_backward_hook(lambda _, gi, go: grads.update(v=go[0].detach()))
    try:
        model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        logits = model(x)
        if target_class is None:
            target_class = int(logits.argmax(1).item())
        logits[0, target_class].backward()
        a, g = acts['v'][0], grads['v'][0]
        cam = torch.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
        cam -= cam.min()
        if cam.max() > 0: cam /= cam.max()
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)
        return cam.squeeze().detach().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

@torch.no_grad()
def attention_rollout(model, x, target_class=None):
    attns, handles = [], []
    for blk in model.blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try: _ = model(x)
    finally:
        for h in handles: h.remove()
    res = None
    for a in attns:
        a = a.mean(1) + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        res = a if res is None else a @ res
    cls = res[0, 0, 1:]; side = int(cls.numel() ** 0.5)
    h = F.interpolate(cls.reshape(side, side)[None, None],
                       size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze().cpu().numpy()
    if h.max() > h.min(): h = (h - h.min()) / (h.max() - h.min())
    return h

def shap_heatmap(model, x, target_class=None, n_samples=60, n_segments=40):
    import shap
    img = x.squeeze(0).permute(1, 2, 0).cpu().numpy()
    seg = slic(img, n_segments=n_segments, compactness=10, start_label=0)
    def f(masks):
        out = np.zeros((masks.shape[0], NUM_CLASSES), dtype=np.float32)
        for i, m in enumerate(masks):
            keep = np.isin(seg, np.where(m == 1)[0])
            masked = img.copy(); masked[~keep] = img.mean(axis=(0, 1))
            t = torch.tensor(masked).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
            with torch.no_grad():
                out[i] = torch.softmax(model(t), 1).cpu().numpy()[0]
        return out
    n_super = int(seg.max() + 1)
    explainer = shap.KernelExplainer(f, np.zeros((1, n_super)))
    sv = explainer.shap_values(np.ones((1, n_super)), nsamples=n_samples)
    if target_class is None:
        with torch.no_grad():
            target_class = int(model(x).argmax(1).item())
    sv = sv[target_class][0]
    heat = np.zeros(seg.shape, dtype=np.float32)
    for sid, val in enumerate(sv):
        heat[seg == sid] = val
    if heat.max() > heat.min():
        heat = (heat - heat.min()) / (heat.max() - heat.min())
    return heat

In [ ]:
@torch.no_grad()
def _prob_for(model, x, cls): return torch.softmax(model(x), 1)[0, cls].item()

def deletion_auc(model, x, heatmap, target_class, steps=30):
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    mean_pix = x.mean(dim=(2, 3), keepdim=True)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = x.clone()
        x_mod[0, :, rows, cols] = mean_pix.expand_as(x)[0, :, rows, cols]
        probs.append(_prob_for(model, x_mod, target_class))
    return float(np.trapezoid(probs, dx=1/steps))

def insertion_auc(model, x, heatmap, target_class, steps=30):
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    base = F.avg_pool2d(x, 15, stride=1, padding=7)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = base.clone(); x_mod[0, :, rows, cols] = x[0, :, rows, cols]
        probs.append(_prob_for(model, x_mod, target_class))
    return float(np.trapezoid(probs, dx=1/steps))

def stability_spearman(h1, h2):
    rho, _ = spearmanr(h1.flatten(), h2.flatten())
    return float(rho) if rho == rho else 0.0

def localization_iou(heatmap, mask, percentile=80):
    if mask is None: return float('nan')
    thr  = np.percentile(heatmap, percentile)
    pred = heatmap >= thr
    inter = np.logical_and(pred, mask).sum()
    union = np.logical_or(pred, mask).sum()
    return float(inter/union) if union > 0 else float('nan')

**DATA ENGINEERING**

In [ ]:
P1 = PHASE_DIRS['phase1_data_engineering']

# 1.1 V2 PATCH — phantom EyeQ filter removed.
# The original filter_pristine() tried to join EyeQ quality labels onto APTOS
# but the join key never matched in any observed Colab run — the routine
# silently fell back to "use all APTOS rows". To keep the methodology honest
# we now explicitly use the full APTOS dataset and document "APTOS only" in
# the dissertation. EyeQ is still loaded separately for the Phase 5 quality
# classifier training (Q_CKPT).
def filter_pristine(aptos_csv, eyeq_csv=None, out_csv=None, **kwargs):
    """V2 patch: always returns the full APTOS dataset."""
    aptos = pd.read_csv(aptos_csv)
    if out_csv is not None:
        aptos.to_csv(out_csv, index=False)
    return aptos

PRISTINE_CSV = P1 / 'metrics' / 'pristine_split.csv'
pristine_df = filter_pristine(APTOS_CSV, EYEQ_CSV, PRISTINE_CSV)
print(f'Using full APTOS dataset: {len(pristine_df)} images')
pristine_df.head()


In [ ]:
# Diagnose the actual APTOS image layout
print('APTOS_IMAGES :', APTOS_IMAGES)
print('Exists?       :', APTOS_IMAGES.exists())
all_files = list(APTOS_IMAGES.rglob('*'))
img_files = [p for p in all_files if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
print(f'Total files under APTOS_IMAGES (any depth): {len(all_files)}')
print(f'Image files found                          : {len(img_files)}')
if img_files:
    print('Sample image paths:')
    for p in img_files[:5]:
        print(' ', p.relative_to(APTOS_IMAGES))
print('\nSample id_code from CSV:', pristine_df["id_code"].iloc[0])
print('Sample image stem      :', img_files[0].stem if img_files else '(none)')

In [ ]:
# Build a stem -> path index so we don't care about extension or subfolder depth.
# Searches all of APTOS_DIR (not just APTOS_IMAGES) in case auto-detection picked the wrong folder.
print('Indexing APTOS images...')
IMAGE_INDEX = {}
for p in APTOS_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.tif', '.tiff'):
        IMAGE_INDEX[p.stem] = p
print(f'Indexed {len(IMAGE_INDEX)} images.')

# Sanity check against the CSV
csv_ids = pristine_df['id_code'].astype(str).tolist()
matched = [i for i in csv_ids if i in IMAGE_INDEX]
missing = [i for i in csv_ids if i not in IMAGE_INDEX]
print(f'CSV ids matched: {len(matched)}/{len(csv_ids)}')
if missing:
    print(f'Example missing id: {missing[0]}')
    print('Example indexed stems:', list(IMAGE_INDEX)[:3])
    raise SystemExit('CSV ids do not match image filenames — inspect the diagnostic output above.')

def resolve_image(id_code):
    """Get the actual image path for an APTOS id_code, regardless of extension or folder."""
    p = IMAGE_INDEX.get(str(id_code))
    if p is None:
        raise FileNotFoundError(f'No image indexed for id_code={id_code!r}')
    return p

In [ ]:
# 1.2 Class-balance plot
counts = pristine_df['diagnosis'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color='steelblue')
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v, str(v), ha='center', va='bottom', fontsize=9)
ax.set_title('Pristine class distribution'); ax.set_ylabel('count')
plt.xticks(rotation=15); plt.tight_layout()
out = P1 / 'plots' / 'class_distribution.png'
plt.savefig(out, dpi=150); plt.show()
print('Saved:', out)

In [ ]:
# 1.3 Synthetic degradation primitives
def _np(img):  return np.array(img.convert('RGB'))
def _pil(arr): return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def gaussian_blur(img, sigma):
    k = max(3, int(2 * round(3 * sigma) + 1))
    return _pil(cv2.GaussianBlur(_np(img), (k, k), sigmaX=sigma, sigmaY=sigma))

def exposure_shift(img, gain): return _pil(_np(img).astype(np.float32) * gain)
def gaussian_noise(img, std_frac):
    arr = _np(img).astype(np.float32)
    return _pil(arr + np.random.normal(0, std_frac * 255, arr.shape))

DEGRADERS = {'blur': gaussian_blur, 'exposure': exposure_shift, 'noise': gaussian_noise}
def apply_degradation(img, kind, level):
    return DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])

In [ ]:
!du -sh /content/data/* 2>/dev/null
!df -h /content

In [ ]:
import shutil

# Use rm -rf for a more forceful deletion to handle potential disk full issues
# or lingering file handles.
!rm -rf {DEGRADED_DIR}

# Ensure the base directory exists before recreating subdirectories.
DEGRADED_DIR.mkdir(parents=True, exist_ok=True)

# Check disk space after deletion to confirm space is freed.
!df -h /content

In [ ]:
# [RESUME GUARD] Skip if degraded images already exist on local disk or Drive cache.
import shutil as _sh

_degraded_present = (
    DEGRADED_DIR.exists()
    and any((DEGRADED_DIR / k / l / "manifest.csv").exists()
            for k in DEGRADATION_TYPES for l in DEGRADATION_LEVELS)
)

# Auto-restore from Drive cache if local disk is empty but cache exists
_deg_cache = DRIVE_ROOT / "cache_degraded.tar.gz"
if not _degraded_present and _deg_cache.exists():
    print(f"[restore] Pulling degraded cache from Drive: {_deg_cache}")
    !tar -xzf "{_deg_cache}" -C /content/data
    _degraded_present = any((DEGRADED_DIR / k / l / "manifest.csv").exists()
                             for k in DEGRADATION_TYPES for l in DEGRADATION_LEVELS)

if _degraded_present:
    print("[skip] degraded images already present — not regenerating.")
    # Still need to make pristine_df available downstream (loaded above in cell 16/17).
else:
    print("[run] Generating degraded sets (this takes ~25 min on A100)...")

# 1.4 (parallel + resized + JPEG) — ~50-100x smaller on disk
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

MAX_WORKERS = max(4, (os.cpu_count() or 2) * 2)
print(f'Using {MAX_WORKERS} worker threads')

DEG_SAVE_SIZE    = (IMAGE_SIZE, IMAGE_SIZE)   # 224x224 — same as model input
DEG_SAVE_FORMAT  = 'JPEG'                      # huge size win vs PNG
DEG_SAVE_QUALITY = 90                          # visually indistinguishable for retinal images
DEG_SAVE_EXT     = '.jpg'

def _process_one(row, kind, level, out_dir):
    try:
        src = resolve_image(row['id_code'])
    except FileNotFoundError:
        return None
    img = Image.open(src).convert('RGB').resize(DEG_SAVE_SIZE, Image.BILINEAR)
    out_img = apply_degradation(img, kind, level)
    rel = f"{row['id_code']}{DEG_SAVE_EXT}"
    out_img.save(out_dir / rel, format=DEG_SAVE_FORMAT, quality=DEG_SAVE_QUALITY, optimize=False)
    return {'id_code': row['id_code'], 'diagnosis': int(row['diagnosis']),
            'degradation': kind, 'level': level, 'rel_path': rel}

def build_split_parallel(kind, level):
    out_dir = DEGRADED_DIR / kind / level
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(_process_one, r, kind, level, out_dir)
                   for _, r in pristine_df.iterrows()]
        for fut in tqdm(as_completed(futures), total=len(futures),
                         desc=f'{kind}-{level}', leave=False):
            r = fut.result()
            if r is not None:
                rows.append(r)
    df = pd.DataFrame(rows)
    df.to_csv(out_dir / 'manifest.csv', index=False)
    df.to_csv(P1 / 'metrics' / f'manifest_{kind}_{level}.csv', index=False)
    return df

for k in DEGRADATION_TYPES:
    for l in DEGRADATION_LEVELS:
        m = build_split_parallel(k, l)
        print(f'  {k}/{l}: {len(m)} images')

!df -h /content"

In [ ]:
# 1.5 Sample grids — extension-agnostic
rng = np.random.default_rng(SEED)
sample_ids = rng.choice(pristine_df['id_code'].values, size=4, replace=False)
for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(4, 4, figsize=(11, 11))
    for r, sid in enumerate(sample_ids):
        clean = Image.open(resolve_image(sid)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        axes[r, 0].imshow(clean); axes[r, 0].set_title('clean' if r == 0 else '')
        for c, lvl in enumerate(DEGRADATION_LEVELS, start=1):
            axes[r, c].imshow(apply_degradation(clean, kind, lvl))
            if r == 0: axes[r, c].set_title(lvl)
        for ax in axes[r]: ax.axis('off')
    fig.suptitle(f'Degradation: {kind}', fontsize=14); plt.tight_layout()
    out = P1 / 'samples' / f'grid_{kind}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print('Saved:', out)

# 1.6 Phase summary
summary = pd.DataFrame([{'degradation': k, 'level': l,
                          'n_images': len(list((DEGRADED_DIR/k/l).glob('*.png')))}
                         for k in DEGRADATION_TYPES for l in DEGRADATION_LEVELS])
summary.to_csv(P1 / 'metrics' / 'phase1_summary.csv', index=False)
summary

In [ ]:
# 2.1 (improved) — stratified split, per-class weights, balanced sampler
P2 = PHASE_DIRS['phase2_model_benchmarking']
PRISTINE_CSV = PHASE_DIRS['phase1_data_engineering'] / 'metrics' / 'pristine_split.csv'

# Build the dataset once with eval transforms; we'll override transforms per-loader.
full_ds_eval  = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, TFM_EVAL)
full_ds_train = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, TFM_TRAIN_STRONG)

labels_all = [int(full_ds_eval.df.iloc[i]['diagnosis']) for i in range(len(full_ds_eval))]
tr_idx, va_idx, te_idx = stratified_split_indices(labels_all)

train_ds = Subset(full_ds_train, tr_idx)
val_ds   = Subset(full_ds_eval,  va_idx)
test_ds  = Subset(full_ds_eval,  te_idx)

train_labels = [labels_all[i] for i in tr_idx]
CLS_W   = class_weights_tensor(train_labels).to(DEVICE)
sampler = balanced_sampler(train_labels)

print('Train class distribution :', Counter(train_labels))
print('Val   class distribution :', Counter([labels_all[i] for i in va_idx]))
print('Test  class distribution :', Counter([labels_all[i] for i in te_idx]))
print('Class weights (loss)     :', CLS_W.cpu().tolist())

common = dict(num_workers=2, pin_memory=True)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, **common)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,   **common)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False,   **common)

# Save test ids for downstream phases
test_ids = [full_ds_eval.df.iloc[i]['id_code'] for i in te_idx]
pd.DataFrame({'id_code': test_ids}).to_csv(P2 / 'metrics' / 'test_ids.csv', index=False)
print(f'\ntrain={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

In [ ]:
def train_v2(name, train_loader, val_loader, ckpt_path, history_csv, hp, class_weights):
    model = build_model(name, pretrained=True).to(DEVICE)

    # Optimizer — LRD for ViT, regular AdamW for CNNs
    if hp['llrd'] is not None and name == 'vit_base':
        param_groups = vit_lrd_param_groups(model, base_lr=hp['lr'],
                                             decay=hp['llrd'], weight_decay=hp['wd'])
        optim = torch.optim.AdamW(param_groups)
    else:
        optim = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])

    sched = warmup_cosine(optim, warmup_epochs=hp['warmup'],
                           total_epochs=hp['epochs'], steps_per_epoch=len(train_loader))
    crit  = FocalLoss(gamma=hp['gamma'], alpha=class_weights, label_smoothing=hp['ls'])
    scaler = GradScaler(enabled=True)
    mixup_fn = Mixup(mixup_alpha=hp['mixup'], cutmix_alpha=1.0,
                      label_smoothing=hp['ls'], num_classes=NUM_CLASSES)
    ema = EMA(model, decay=0.999)

    best_f1, history = -1.0, []
    for ep in range(hp['epochs']):
        model.train(); running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f'[{name}] ep {ep+1}/{hp["epochs"]}', leave=False)
        for x, y, _ in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            x, y_soft = mixup_fn(x, y)
            optim.zero_grad(set_to_none=True)
            with autocast(enabled=True):
                loss = crit(model(x), y_soft)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step(); ema.update(model)
            running += loss.item() * x.size(0); seen += x.size(0)
            pbar.set_postfix(loss=running/seen, lr=optim.param_groups[0]['lr'])

        # Evaluate with EMA weights — usually a bit better
        with ema.applied(model):
            m = evaluate(model, val_loader)
        history.append({'epoch': ep + 1, 'train_loss': running / seen, **m,
                        'lr': optim.param_groups[0]['lr']})
        print(f'  val: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}  auc={m["auc_macro_ovr"]:.4f}')
        if m['f1_macro'] > best_f1:
            best_f1 = m['f1_macro']
            with ema.applied(model):
                torch.save({'state_dict': {k: v.clone() for k, v in model.state_dict().items()},
                            'arch': name, 'val': m, 'hp': hp}, ckpt_path)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    return pd.DataFrame(history)

print('train_v2 ready.')

In [ ]:
# [RESUME GUARD] Skip v1/v2 training if all checkpoints exist on Drive.
_v2_present = all((CHECKPOINTS_DIR / f"{n}_best.pt").exists() for n in MODEL_NAMES)
if _v2_present:
    print("[skip] v2 checkpoints already on Drive — not retraining.")
    print("        Found:", *[f"{n}_best.pt" for n in MODEL_NAMES])
    histories = {}   # placeholder so downstream cells don't NameError
else:
    print("[run] Training v1/v2 models (this takes ~2 hours on A100)...")
    histories = {}
    for name in MODEL_NAMES:
        print(f'\n========== Training {name} ==========')
        hp = HPARAMS[name]
        histories[name] = train_v2(name, train_loader, val_loader,
                                    ckpt_path=CHECKPOINTS_DIR / f'{name}_best.pt',
                                    history_csv=P2 / 'metrics' / f'training_history_{name}.csv',
                                    hp=hp, class_weights=CLS_W)
        torch.cuda.empty_cache()

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
# 2.3 Training curves
for name, h in histories.items():
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h['epoch'], h['train_loss']); ax[0].set_title(f'{name} — train loss')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss')
    ax[1].plot(h['epoch'], h['accuracy'], label='val acc')
    ax[1].plot(h['epoch'], h['f1_macro'], label='val F1')
    ax[1].set_title(f'{name} — val metrics'); ax[1].legend(); ax[1].set_xlabel('epoch')
    plt.tight_layout()
    out = P2 / 'plots' / f'training_curves_{name}.png'
    plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)

In [ ]:
@torch.no_grad()
def evaluate_tta(model, loader, n_aug=4):
    """Test-time augmentation: average predictions over hflip/vflip variants."""
    model.eval(); ys, probs = [], []
    augs = [lambda x: x, torch.flip, lambda x: torch.flip(x, [3]),
            lambda x: torch.flip(x, [2])][:n_aug]
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        avg = None
        for fn in augs:
            xa = fn(x) if fn is not torch.flip else torch.flip(x, [2, 3])
            p = torch.softmax(model(xa), 1)
            avg = p if avg is None else avg + p
        avg /= len(augs)
        probs.append(avg.cpu().numpy()); ys.append(y.numpy())
    y = np.concatenate(ys); pr = np.concatenate(probs); p = pr.argmax(1)
    out = {'accuracy': accuracy_score(y, p), 'f1_macro': f1_score(y, p, average='macro')}
    try: out['auc_macro_ovr'] = roc_auc_score(y, pr, average='macro', multi_class='ovr')
    except ValueError: out['auc_macro_ovr'] = float('nan')
    return out

test_id_set = set(map(str, test_ids))
def deg_loader(kind, level):
    ds = FolderDataset(DEGRADED_DIR / kind / level)
    ds.df = ds.df[ds.df['id_code'].astype(str).isin(test_id_set)].reset_index(drop=True)
    return DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

rows = []
for name in MODEL_NAMES:
    print(f'\n--- {name} (TTA) ---')
    model = load_classifier(name)
    m = evaluate_tta(model, test_loader)
    rows.append({'model': name, 'degradation': 'clean', 'level': 'none', **m})
    print(f'  clean: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}')
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            m = evaluate_tta(model, deg_loader(k, l))
            rows.append({'model': name, 'degradation': k, 'level': l, **m})
            print(f'  {k}/{l}: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}')
    del model; torch.cuda.empty_cache()

stress_df = pd.DataFrame(rows)
stress_df.to_csv(P2 / 'metrics' / 'stress_test_results.csv', index=False)
pivot = stress_df.pivot_table(index=['degradation', 'level'], columns='model', values='accuracy').round(3)
pivot.to_csv(P2 / 'metrics' / 'accuracy_pivot.csv')
pivot

In [ ]:
# [RESUME] Optuna hyper-parameter search already done -- skipped.
print('[resume] skipped Optuna HPO (using existing checkpoints/HPARAMS).')


In [ ]:
# Ben Graham fundus preprocessing — circle-crop + local-mean subtraction
import cv2

def _circle_crop(arr_rgb):
    h, w = arr_rgb.shape[:2]
    gray = cv2.cvtColor(arr_rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 7, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr_rgb
    c = max(contours, key=cv2.contourArea)
    x, y, ww, hh = cv2.boundingRect(c)
    if ww < 50 or hh < 50:
        return arr_rgb
    return arr_rgb[y:y+hh, x:x+ww]


class BenGraham:
    """Ben Graham's APTOS-winning preprocessing.

    1. Crop the black border around the fundus disc.
    2. Subtract a Gaussian-blurred copy and re-centre.
       -> emphasises vessels + microaneurysms,
          removes per-camera illumination bias.
    """
    def __init__(self, sigma_x=10):
        self.sigma_x = sigma_x

    def __call__(self, img_pil):
        arr  = np.array(img_pil.convert("RGB")).astype(np.float32)
        arr  = _circle_crop(arr).astype(np.float32)
        blur = cv2.GaussianBlur(arr, (0, 0), self.sigma_x)
        out  = cv2.addWeighted(arr, 4, blur, -4, 128)
        return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))


# Quick visual sanity check on one sample
_demo_id = pristine_df["id_code"].iloc[0]
_orig    = Image.open(resolve_image(_demo_id)).convert("RGB").resize((300, 300))
_bg      = BenGraham()(_orig)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(_orig); axes[0].set_title("original"); axes[0].axis("off")
axes[1].imshow(_bg);   axes[1].set_title("Ben Graham");  axes[1].axis("off")
plt.tight_layout(); plt.show()
print("Ben Graham preprocessor ready.")


In [ ]:
IMAGE_SIZE_V3 = 384

TFM_TRAIN_V3 = T.Compose([
    BenGraham(sigma_x=10),
    T.Resize((IMAGE_SIZE_V3 + 32, IMAGE_SIZE_V3 + 32)),
    T.RandomCrop(IMAGE_SIZE_V3),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(25),
    T.RandAugment(num_ops=2, magnitude=5),
    T.ColorJitter(0.10, 0.10, 0.10, 0.02),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.20),
])

TFM_EVAL_V3 = T.Compose([
    BenGraham(sigma_x=10),
    T.Resize((IMAGE_SIZE_V3, IMAGE_SIZE_V3)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print(f"V3 transforms ready (input {IMAGE_SIZE_V3}px).")

In [ ]:
class MultiScaleClassifier(nn.Module):
    """Wrap any timm backbone with multi-stage feature fusion.

    forward(x)                      -> logits
    forward(x, return_ordinal=True) -> (logits, ordinal_score)

    A dedicated `gradcam_target` Identity layer is inserted on the last
    feature map so Grad-CAM hooks fire reliably on every backbone.
    """
    def __init__(self, backbone_name, num_classes=5, pretrained=True,
                 dropout=0.3, image_size=384, n_stages=4):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained, features_only=True,
        )

        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, image_size, image_size)
            feats_full = self.backbone(dummy)
        if was_training:
            self.backbone.train()

        self.n_stages = min(n_stages, len(feats_full))
        feat_dims = [f.shape[1] for f in feats_full[-self.n_stages:]]
        total_dim = sum(feat_dims)

        self._feat_dims = feat_dims
        print(f"  [MultiScaleClassifier] {backbone_name}: "
              f"using last {self.n_stages} of {len(feats_full)} stages, "
              f"dims={feat_dims}, total={total_dim}")

        self.stage_attn = nn.Parameter(torch.ones(self.n_stages))
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.fusion = nn.Sequential(
            nn.Linear(total_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        self.ordinal_head = nn.Linear(total_dim, 1)

        # ---- Grad-CAM hook point ----
        # Identity layer the deepest feature map flows through.
        # Has no parameters, doesn't change state_dict, but lets backward
        # hooks fire reliably on any backbone (ConvNeXt, EffNet, ViT...).
        self.gradcam_target = nn.Identity()

        self._dr_arch = backbone_name

    def forward_features(self, x):
        feats = list(self.backbone(x))[-self.n_stages:]
        feats[-1] = self.gradcam_target(feats[-1])           # <-- Grad-CAM hook point
        attn   = torch.softmax(self.stage_attn, dim=0)
        pooled = [self.pool(f).flatten(1) * attn[i] for i, f in enumerate(feats)]
        return torch.cat(pooled, dim=1)

    def forward(self, x, return_ordinal=False):
        z = self.forward_features(x)
        logits = self.fusion(z)
        if return_ordinal:
            return logits, self.ordinal_head(z).squeeze(-1)
        return logits


print("MultiScaleClassifier ready (with gradcam_target Identity hook).")

In [ ]:
class OrdinalFocalLoss(nn.Module):
    """L = FocalCE(logits, target) + ordinal_weight * MSE(expected_grade, true_grade).

    Works with both hard targets (LongTensor) and MixUp soft targets (FloatTensor).
    """
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.05,
                 ordinal_weight=0.3, num_classes=5):
        super().__init__()
        self.focal = FocalLoss(gamma=gamma, alpha=alpha, label_smoothing=label_smoothing)
        self.ord_w = ordinal_weight
        self.k     = num_classes

    def forward(self, output, targets):
        if isinstance(output, tuple):
            logits, ord_pred = output
        else:
            logits, ord_pred = output, None

        # ----- classification term -----
        if targets.dim() > 1:
            log_p    = F.log_softmax(logits, dim=-1)
            cls_loss = -(targets * log_p).sum(-1).mean()
        else:
            cls_loss = self.focal(logits, targets)

        if ord_pred is None:
            return cls_loss

        # ----- ordinal term -----
        grades          = torch.arange(self.k, device=logits.device, dtype=torch.float)
        soft_grade_pred = (torch.softmax(logits, dim=-1) * grades).sum(-1)
        true_grade      = (targets * grades).sum(-1) if targets.dim() > 1 else targets.float()
        ord_loss        = F.mse_loss(soft_grade_pred, true_grade)
        return cls_loss + self.ord_w * ord_loss


print("OrdinalFocalLoss ready.")


In [ ]:
from sklearn.metrics import cohen_kappa_score

BACKBONE_V3 = {
    "resnet50":        "convnext_base.fb_in22k_ft_in1k_384",
    "efficientnet_b3": "tf_efficientnetv2_s.in21k_ft_in1k",
    "vit_base":        "vit_base_patch16_clip_384.laion2b_ft_in12k_in1k",
}

HPARAMS_V3 = {
    "resnet50":        dict(lr=1e-4, wd=1e-4, epochs=20, mixup=0.10, ls=0.05,
                             gamma=1.5, warmup=2, llrd=None, ord_w=0.30),
    "efficientnet_b3": dict(lr=1e-4, wd=1e-5, epochs=20, mixup=0.10, ls=0.05,
                             gamma=2.0, warmup=2, llrd=None, ord_w=0.30),
    "vit_base":        dict(lr=3e-5, wd=5e-2, epochs=25, mixup=0.20, ls=0.05,
                             gamma=2.0, warmup=4, llrd=0.75, ord_w=0.30),
}


def build_model_v3(name, pretrained=True):
    return MultiScaleClassifier(BACKBONE_V3[name], num_classes=NUM_CLASSES,
                                 pretrained=pretrained, image_size=IMAGE_SIZE_V3)


@torch.no_grad()
def evaluate_v3(model, loader):
    model.eval(); ys, probs = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs.append(torch.softmax(logits, 1).cpu().numpy())
        ys.append(y.numpy())
    y  = np.concatenate(ys)
    pr = np.concatenate(probs)
    p  = pr.argmax(1)
    out = {
        "accuracy": accuracy_score(y, p),
        "f1_macro": f1_score(y, p, average="macro"),
        "kappa_qw": cohen_kappa_score(y, p, weights="quadratic"),
    }
    try:
        out["auc_macro_ovr"] = roc_auc_score(y, pr, average="macro", multi_class="ovr")
    except ValueError:
        out["auc_macro_ovr"] = float("nan")
    return out


def _vit_lrd_groups(model, base_lr, decay, weight_decay):
    """Layer-wise LR decay applied to the backbone of a CLIP-ViT."""
    n_layers = 13   # CLIP ViT-B has 12 blocks + head
    groups = []
    for n, p in model.backbone.named_parameters():
        if not p.requires_grad: continue
        if "blocks" in n:
            try:
                depth = int([t for t in n.split(".") if t.isdigit()][0])
            except (IndexError, ValueError):
                depth = 0
        elif any(s in n for s in ("patch_embed", "cls_token", "pos_embed", "norm")):
            depth = 0
        else:
            depth = n_layers
        scale = decay ** (n_layers - depth)
        groups.append({"params": p, "lr": base_lr * scale,
                       "weight_decay": 0.0 if p.ndim <= 1 else weight_decay})
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    groups.append({"params": head_params, "lr": base_lr, "weight_decay": weight_decay})
    return groups


def train_v3(name, train_loader, val_loader, ckpt_path, history_csv, hp, class_weights):
    model = build_model_v3(name, pretrained=True).to(DEVICE)

    if hp["llrd"] is not None and "vit" in BACKBONE_V3[name]:
        optim = torch.optim.AdamW(_vit_lrd_groups(model, hp["lr"], hp["llrd"], hp["wd"]))
    else:
        optim = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["wd"])

    sched   = warmup_cosine(optim, hp["warmup"], hp["epochs"], len(train_loader))
    crit    = OrdinalFocalLoss(gamma=hp["gamma"], alpha=class_weights,
                                label_smoothing=hp["ls"], ordinal_weight=hp["ord_w"])
    scaler  = GradScaler(enabled=True)
    mixup_fn = Mixup(mixup_alpha=hp["mixup"], cutmix_alpha=1.0,
                      label_smoothing=hp["ls"], num_classes=NUM_CLASSES)
    ema     = EMA(model, decay=0.999)

    best_kappa, history = -1.0, []
    for ep in range(hp["epochs"]):
        model.train(); running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f"[{name}] ep {ep+1}/{hp['epochs']}", leave=False)
        for x, y, _ in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            x, y_soft = mixup_fn(x, y)
            optim.zero_grad(set_to_none=True)
            with autocast(enabled=True):
                logits, ord_pred = model(x, return_ordinal=True)
                loss = crit((logits, ord_pred), y_soft)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step(); ema.update(model)
            running += loss.item() * x.size(0); seen += x.size(0)
            pbar.set_postfix(loss=running/seen, lr=optim.param_groups[0]["lr"])

        with ema.applied(model):
            m = evaluate_v3(model, val_loader)
        history.append({"epoch": ep+1, "train_loss": running/seen, **m,
                         "lr": optim.param_groups[0]["lr"]})
        print(f"  val: acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  "
              f"kappa={m['kappa_qw']:.4f}  auc={m['auc_macro_ovr']:.4f}")
        if m["kappa_qw"] > best_kappa:
            best_kappa = m["kappa_qw"]
            with ema.applied(model):
                torch.save({"state_dict": {k: v.clone() for k, v in model.state_dict().items()},
                            "arch": name, "backbone": BACKBONE_V3[name],
                            "val": m, "hp": hp}, ckpt_path)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    return pd.DataFrame(history)


print("train_v3 / evaluate_v3 / build_model_v3 ready.")


In [ ]:
# Rebuild loaders at 384 with Ben Graham preprocessing
full_eval_v3  = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, TFM_EVAL_V3)
full_train_v3 = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, TFM_TRAIN_V3)

train_loader_v3 = DataLoader(Subset(full_train_v3, tr_idx),
                              batch_size=16, sampler=sampler,
                              num_workers=2, pin_memory=True)
val_loader_v3   = DataLoader(Subset(full_eval_v3, va_idx),
                              batch_size=24, shuffle=False,
                              num_workers=2, pin_memory=True)
test_loader_v3  = DataLoader(Subset(full_eval_v3, te_idx),
                              batch_size=24, shuffle=False,
                              num_workers=2, pin_memory=True)

CKPT_DIR_V3 = CHECKPOINTS_DIR / "v3"
CKPT_DIR_V3.mkdir(exist_ok=True)
HIST_DIR_V3 = P2 / "metrics" / "v3"
HIST_DIR_V3.mkdir(exist_ok=True)

print(f"V3 loaders ready. train={len(tr_idx)}  val={len(va_idx)}  test={len(te_idx)}")
print(f"Checkpoints -> {CKPT_DIR_V3}")
print(f"History     -> {HIST_DIR_V3}")
# [RESUME GUARD] Skip v3 training if all v3 checkpoints exist on Drive.
_v3_present = all((CKPT_DIR_V3 / f"{n}_v3_best.pt").exists() for n in MODEL_NAMES)
if _v3_present:
    print("[skip] v3 checkpoints already on Drive — not retraining.")
    print("        Found:", *[f"{n}_v3_best.pt" for n in MODEL_NAMES])
    histories_v3 = {}   # placeholder so downstream cells don't NameError
else:
    print("[run] Training v3 models (this takes ~3 hours on A100)...")
    histories_v3 = {}
    for name in MODEL_NAMES:
        ckpt = CKPT_DIR_V3 / f"{name}_v3_best.pt"
        if ckpt.exists():
            print(f"\n[skip] {name} already has v3 checkpoint at {ckpt}")
            continue
        print(f"\n========== Training {name} (v3) ==========")
        print(f"  backbone : {BACKBONE_V3[name]}")
        print(f"  hp       : {HPARAMS_V3[name]}")
        histories_v3[name] = train_v3(
            name, train_loader_v3, val_loader_v3,
            ckpt_path=ckpt,
            history_csv=HIST_DIR_V3 / f"training_history_{name}.csv",
            hp=HPARAMS_V3[name], class_weights=CLS_W,
        )
        torch.cuda.empty_cache()

In [ ]:
def load_classifier_v3(name):
    """Load a v3 multi-scale classifier from disk.

    V2 PATCH: prefers `{name}_v3_best.pt` (fully-trained) over the undertrained
    `{name}_v3_distilled.pt`. The original V4 distillation run was interrupted
    at epoch 3 of the resnet50 student, leaving two ~3-epoch distilled
    checkpoints on Drive. The previous prefer-distilled logic silently routed
    those under-trained students into the V4 ensemble and clean accuracy
    regressed 0.814 -> 0.725. Set USE_DISTILLED = True below only after you
    have re-run distillation to the full 10 epochs and verified val kappa.
    """
    USE_DISTILLED = False   # flip back to True once distillation reruns to convergence
    m = build_model_v3(name, pretrained=False).to(DEVICE)
    distilled = CKPT_DIR_V3 / f"{name}_v3_distilled.pt"
    best      = CKPT_DIR_V3 / f"{name}_v3_best.pt"
    src = distilled if (USE_DISTILLED and distilled.exists()) else best
    ckpt = torch.load(src, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt["state_dict"]); m.eval()
    return m


@torch.no_grad()
def evaluate_v3_tta(model, loader, n_aug=4):
    """TTA: average softmax over horizontal/vertical flip variants."""
    model.eval(); ys, probs = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        views = [x,
                 torch.flip(x, [3]),
                 torch.flip(x, [2]),
                 torch.flip(x, [2, 3])][:n_aug]
        avg = None
        for v in views:
            p = torch.softmax(model(v), 1)
            avg = p if avg is None else avg + p
        avg /= len(views)
        probs.append(avg.cpu().numpy()); ys.append(y.numpy())
    y  = np.concatenate(ys)
    pr = np.concatenate(probs)
    p  = pr.argmax(1)
    out = {
        "accuracy": accuracy_score(y, p),
        "f1_macro": f1_score(y, p, average="macro"),
        "kappa_qw": cohen_kappa_score(y, p, weights="quadratic"),
    }
    try:
        out["auc_macro_ovr"] = roc_auc_score(y, pr, average="macro", multi_class="ovr")
    except ValueError:
        out["auc_macro_ovr"] = float("nan")
    return out


# Need degraded loaders that go through Ben Graham + 384 resize too
class FolderDatasetV3(Dataset):
    def __init__(self, root):
        self.root = Path(root); self.df = pd.read_csv(self.root / "manifest.csv")
        self.transform = TFM_EVAL_V3
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row["rel_path"]).convert("RGB")
        return self.transform(img), int(row["diagnosis"]), row["id_code"]


test_id_set = set(map(str, [full_eval_v3.df.iloc[i]["id_code"] for i in te_idx]))


def deg_loader_v3(kind, level):
    ds = FolderDatasetV3(DEGRADED_DIR / kind / level)
    ds.df = ds.df[ds.df["id_code"].astype(str).isin(test_id_set)].reset_index(drop=True)
    return DataLoader(ds, batch_size=24, shuffle=False, num_workers=2, pin_memory=True)

# [RESUME GUARD] Skip v3 stress-test loop if the saved CSV already exists.
_v3_stress_csv = P2 / "metrics" / "v3" / "stress_test_results_v3.csv"
if _v3_stress_csv.exists():
    print(f"[skip] v3 stress-test CSV already exists -> {_v3_stress_csv}")
    stress_df_v3 = pd.read_csv(_v3_stress_csv)
    pivot_v3 = stress_df_v3.pivot_table(index=["degradation", "level"],
                                         columns="model", values="accuracy").round(3)
    pivot_v3
else:
    print("[run] Running v3 stress test (this takes ~30 min on A100)...")
    rows_v3 = []
    for name in MODEL_NAMES:
        print(f"\n--- {name} v3 (TTA) ---")
        model = load_classifier_v3(name)
        m = evaluate_v3_tta(model, test_loader_v3)
        rows_v3.append({"model": name, "degradation": "clean", "level": "none", **m})
        print(f"  clean: acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  kappa={m['kappa_qw']:.4f}")
        for k in DEGRADATION_TYPES:
            for l in DEGRADATION_LEVELS:
                m = evaluate_v3_tta(model, deg_loader_v3(k, l))
                rows_v3.append({"model": name, "degradation": k, "level": l, **m})
                print(f"  {k}/{l}: acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  kappa={m['kappa_qw']:.4f}")
        del model; torch.cuda.empty_cache()
    stress_df_v3 = pd.DataFrame(rows_v3)
    stress_df_v3.to_csv(_v3_stress_csv, index=False)
    pivot_v3 = stress_df_v3.pivot_table(index=["degradation", "level"],
                                         columns="model", values="accuracy").round(3)
    pivot_v3


## V4 - 8-view TTA + 3-model soft-vote ensemble

After v3 training finishes, this cell runs an ensemble evaluation of all three backbones with 8-view TTA on top. Typically +1 to +2 accuracy points on top of single-model TTA results, completely free at inference. Results saved to `results/phase2_model_benchmarking/v3/ensemble_v4_results.csv` for the literature-review comparison.

In [ ]:
# === V4 PATCH: 8-view TTA + 3-model soft-vote ensemble (val-kappa weighted) ===
# Equal-weight voting let the weakest backbone drag down the ensemble — now each
# model's softmax is weighted by its checkpoint's val-kappa, so a strong model
# dominates on the slices it's best at.

@torch.no_grad()
def evaluate_v3_tta8(model, loader):
    """8-view test-time augmentation: identity, 3 flips, 3 rotations, flip+rot."""
    model.eval(); ys, probs = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        views = [
            x,
            torch.flip(x, [3]),
            torch.flip(x, [2]),
            torch.flip(x, [2, 3]),
            torch.rot90(x, 1, [2, 3]),
            torch.rot90(x, 2, [2, 3]),
            torch.rot90(x, 3, [2, 3]),
            torch.flip(torch.rot90(x, 1, [2, 3]), [3]),
        ]
        avg = None
        for v in views:
            p = torch.softmax(model(v), 1)
            avg = p if avg is None else avg + p
        avg /= len(views)
        probs.append(avg.cpu().numpy()); ys.append(y.numpy())
    y  = np.concatenate(ys)
    pr = np.concatenate(probs)
    p  = pr.argmax(1)
    out = {
        "accuracy": accuracy_score(y, p),
        "f1_macro": f1_score(y, p, average="macro"),
        "kappa_qw": cohen_kappa_score(y, p, weights="quadratic"),
    }
    try:
        out["auc_macro_ovr"] = roc_auc_score(y, pr, average="macro", multi_class="ovr")
    except ValueError:
        out["auc_macro_ovr"] = float("nan")
    return out


def _read_val_kappa(name):
    """Return the val kappa saved in the model's checkpoint (distilled preferred)."""
    distilled = CKPT_DIR_V3 / f"{name}_v3_distilled.pt"
    best      = CKPT_DIR_V3 / f"{name}_v3_best.pt"
    src = distilled if distilled.exists() else best
    ckpt = torch.load(src, map_location="cpu", weights_only=False)
    return float(ckpt.get("val", {}).get("kappa_qw", 0.5))


@torch.no_grad()
def evaluate_v3_ensemble(loader, use_tta8=True):
    """Val-kappa-weighted soft-vote across all v3 backbones."""
    models  = {n: load_classifier_v3(n) for n in MODEL_NAMES}
    kappas  = {n: _read_val_kappa(n) for n in MODEL_NAMES}
    # Clamp negative/zero kappas, then renormalise
    weights = {n: max(0.05, kappas[n]) for n in MODEL_NAMES}
    total_w = sum(weights.values())
    weights = {n: weights[n] / total_w for n in MODEL_NAMES}
    print(f"  ensemble weights: " + ", ".join(f"{n}={weights[n]:.2f}" for n in MODEL_NAMES))
    for m in models.values():
        m.eval()
    ys, probs = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        avg = None
        for name, mdl in models.items():
            if use_tta8:
                views = [
                    x, torch.flip(x, [3]), torch.flip(x, [2]),
                    torch.flip(x, [2, 3]),
                    torch.rot90(x, 1, [2, 3]), torch.rot90(x, 2, [2, 3]),
                    torch.rot90(x, 3, [2, 3]),
                ]
                m_pred = sum(torch.softmax(mdl(v), 1) for v in views) / len(views)
            else:
                m_pred = torch.softmax(mdl(x), 1)
            contrib = m_pred * weights[name]
            avg = contrib if avg is None else avg + contrib
        probs.append(avg.cpu().numpy()); ys.append(y.numpy())
    for m in models.values(): del m
    torch.cuda.empty_cache()
    y  = np.concatenate(ys)
    pr = np.concatenate(probs)
    p  = pr.argmax(1)
    out = {
        "accuracy": accuracy_score(y, p),
        "f1_macro": f1_score(y, p, average="macro"),
        "kappa_qw": cohen_kappa_score(y, p, weights="quadratic"),
    }
    try:
        out["auc_macro_ovr"] = roc_auc_score(y, pr, average="macro", multi_class="ovr")
    except ValueError:
        out["auc_macro_ovr"] = float("nan")
    return out


# ----- Run the ensemble on clean + every degraded condition -----
rows_ens = []

print("=== Ensemble (8-TTA, val-kappa weighted) on clean ===")
m = evaluate_v3_ensemble(test_loader_v3, use_tta8=True)
rows_ens.append({"degradation": "clean", "level": "none", **m})
print(f"  acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  kappa={m['kappa_qw']:.4f}")

for k in DEGRADATION_TYPES:
    for l in DEGRADATION_LEVELS:
        m = evaluate_v3_ensemble(deg_loader_v3(k, l), use_tta8=True)
        rows_ens.append({"degradation": k, "level": l, **m})
        print(f"  {k}/{l}: acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  kappa={m['kappa_qw']:.4f}")

ens_df_v4 = pd.DataFrame(rows_ens)
ens_csv = P2 / "metrics" / "v3" / "ensemble_v4_results.csv"
ens_df_v4.to_csv(ens_csv, index=False)
print("Saved:", ens_csv)
ens_df_v4


## V2 vs V3 — head-to-head comparison (literature-review fodder)

Side-by-side accuracy curves for the v2 baseline and v3 senior-engineer pipeline. Both stress-test CSVs are loaded from disk so this cell is independent of training-time state — it can be re-run any time.

In [ ]:
# V2 vs V3 head-to-head: accuracy-vs-degradation curves on the same axes
import matplotlib.pyplot as plt

V2_CSV = P2 / "metrics" / "stress_test_results.csv"
V3_CSV = P2 / "metrics" / "v3" / "stress_test_results_v3.csv"

if not V2_CSV.exists() or not V3_CSV.exists():
    print(f"  v2 csv exists: {V2_CSV.exists()}  v3 csv exists: {V3_CSV.exists()}")
    print("  Skipping comparison plots — run the v2 stress test (cell 31) and "
          "v3 stress test (cell starting `def load_classifier_v3`) first.")
else:
    df_v2 = pd.read_csv(V2_CSV).assign(version="v2")
    df_v3 = pd.read_csv(V3_CSV).assign(version="v3")
    df = pd.concat([df_v2, df_v3], ignore_index=True)

    level_order = ["none", "low", "mid", "high"]
    df["level"] = pd.Categorical(df["level"], categories=level_order, ordered=True)

    save_dir = P2 / "plots" / "v2_vs_v3"
    save_dir.mkdir(parents=True, exist_ok=True)

    for kind in DEGRADATION_TYPES:
        fig, axes = plt.subplots(1, len(MODEL_NAMES),
                                  figsize=(4.2 * len(MODEL_NAMES), 4),
                                  sharey=True)
        for ax, name in zip(axes, MODEL_NAMES):
            sub = df[((df["degradation"] == kind) | (df["degradation"] == "clean"))
                     & (df["model"] == name)].sort_values("level")
            for ver, style in [("v2", dict(linestyle="--", marker="s", alpha=0.75)),
                                ("v3", dict(linestyle="-",  marker="o", alpha=1.00))]:
                d = sub[sub["version"] == ver]
                if not d.empty:
                    ax.plot(d["level"], d["accuracy"], label=ver, **style)
            ax.set_title(name); ax.set_xlabel("level")
            ax.set_ylim(0, 1); ax.grid(alpha=0.3)
        axes[0].set_ylabel("accuracy"); axes[-1].legend()
        fig.suptitle(f"V2 vs V3 — accuracy under {kind}", fontsize=13)
        plt.tight_layout()
        out = save_dir / f"v2_vs_v3_{kind}.png"
        plt.savefig(out, dpi=150); plt.show()
        print("Saved:", out)

    # Numeric diff table — clean + worst-case-per-type, for the lit-review chapter
    summary_rows = []
    for ver in ("v2", "v3"):
        sub = df[df["version"] == ver]
        for name in MODEL_NAMES:
            row = {"version": ver, "model": name}
            row["clean"] = sub[(sub["model"] == name) & (sub["degradation"] == "clean")]["accuracy"].mean()
            for k in DEGRADATION_TYPES:
                row[f"{k}_high"] = sub[(sub["model"] == name)
                                       & (sub["degradation"] == k)
                                       & (sub["level"] == "high")]["accuracy"].mean()
            summary_rows.append(row)
    diff_df = pd.DataFrame(summary_rows).round(3)
    diff_df.to_csv(P2 / "metrics" / "v2_vs_v3_summary.csv", index=False)
    print("\n=== V2 vs V3 headline accuracy ===")
    print(diff_df.to_string(index=False))


In [ ]:
def distill_to(student_name, teacher_name, train_loader, val_loader, ckpt_path,
                T_temp=4.0, alpha=0.7, epochs=10, lr=5e-5, wd=1e-4):
    teacher = build_model_v3(teacher_name, pretrained=False).to(DEVICE)
    teacher.load_state_dict(
    torch.load(CKPT_DIR_V3 / f"{teacher_name}_v3_best.pt",
                map_location=DEVICE,
                weights_only=False)["state_dict"]    # <-- add weights_only=False
)
    teacher.eval()
    student = build_model_v3(student_name, pretrained=True).to(DEVICE)

    optim  = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=wd)
    sched  = warmup_cosine(optim, 1, epochs, len(train_loader))
    scaler = GradScaler(enabled=True)
    ce     = nn.CrossEntropyLoss(label_smoothing=0.05, weight=CLS_W)

    best_kappa = -1.0
    for ep in range(epochs):
        student.train()
        pbar = tqdm(train_loader, desc=f"distill {student_name} ep {ep+1}/{epochs}", leave=False)
        for x, y, _ in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with autocast(enabled=True):
                with torch.no_grad():
                    t_logits = teacher(x) / T_temp
                s_logits = student(x)
                kd_loss = F.kl_div(F.log_softmax(s_logits / T_temp, -1),
                                    F.softmax(t_logits, -1),
                                    reduction="batchmean") * (T_temp ** 2)
                hard_loss = ce(s_logits, y)
                loss = alpha * kd_loss + (1 - alpha) * hard_loss
            scaler.scale(loss).backward()
            scaler.step(optim); scaler.update(); sched.step()
        m = evaluate_v3(student, val_loader)
        print(f"  ep {ep+1}: acc={m['accuracy']:.4f}  kappa={m['kappa_qw']:.4f}")
        if m["kappa_qw"] > best_kappa:
            best_kappa = m["kappa_qw"]
            torch.save({"state_dict": student.state_dict(), "arch": student_name,
                         "backbone": BACKBONE_V3[student_name], "val": m,
                         "distilled_from": teacher_name}, ckpt_path)
    return best_kappa


# --- Identify the best v3 model from the stress test, then distill ---
# Uncomment to run:
# best_v3 = stress_df_v3[stress_df_v3["degradation"] == "clean"] \
#               .sort_values("kappa_qw", ascending=False).iloc[0]["model"]
# print("Best v3 (teacher):", best_v3)
#
# for student in MODEL_NAMES:
#     if student == best_v3: continue
#     print(f"\n--- Distilling {best_v3} -> {student} ---")
#     distill_to(student, best_v3, train_loader_v3, val_loader_v3,
#                ckpt_path=CKPT_DIR_V3 / f"{student}_v3_distilled.pt")


In [ ]:
# [RESUME] Knowledge distillation already done -- skipped.
print('[resume] skipped distillation selection.')


## V4 - Knowledge distillation runner

Picks the best v3 model as the teacher and distills into the other two. Each student trains 10 epochs with a KL-divergence loss against the teacher's softmax at temperature 4. Typical gain: +0.5 to +1.5 kappa on the student. Skip this if Colab time is tight.

In [ ]:
# [RESUME] Knowledge distillation runner already done -- skipped.
print('[resume] skipped distillation training.')


In [ ]:
USE_V3 = True

if USE_V3:
    print("[adapter] Phase 3-5 will use V3 multi-scale classifiers @384.")
    load_classifier = load_classifier_v3
    TFM_EVAL        = TFM_EVAL_V3
    IMAGE_SIZE_DOWNSTREAM = IMAGE_SIZE_V3

    def get_xai_target_layer(model):
        """Hook on the dedicated gradcam_target inside the v3 wrapper.

        Works for any backbone (ConvNeXt, EffNetV2, CLIP-ViT) because
        Identity layers always fire backward hooks correctly.
        """
        if hasattr(model, "gradcam_target"):
            return model.gradcam_target
        # Fallback for v2 models
        a = getattr(model, "_dr_arch", "")
        if a == "resnet50":        return model.layer4[-1]
        if a == "efficientnet_b3": return model.blocks[-1]
        if a == "vit_base":        return model.blocks[-1].norm1
        return list(model.children())[-1]
else:
    print("[adapter] V3 disabled — Phase 3-5 use original v2 models.")

print("Adapter installed.")

In [ ]:
P3 = PHASE_DIRS['phase3_xai_benchmark']

# 3.1 Pick the same N test images we'll explain across every condition
N_EXPLAIN = 20                      # bump for final run; SHAP is the bottleneck
test_ids_full = pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].tolist()
EXPLAIN_IDS = pd.Series(test_ids_full).sample(N_EXPLAIN, random_state=SEED).tolist()
labels_df = pd.read_csv(PRISTINE_CSV).set_index('id_code')['diagnosis']

models = {n: load_classifier(n) for n in MODEL_NAMES}
print('Loaded:', list(models))

# Optional ground-truth lesion masks at /content/data/lesion_masks/<id>.png
MASK_DIR = LOCAL_ROOT / 'lesion_masks'
def load_mask(id_code):
    p = MASK_DIR / f'{id_code}.png'
    if not p.exists(): return None
    return (np.array(Image.open(p).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE))) > 127)

## V4 - Integrated Gradients in place of KernelSHAP

KernelSHAP at the original sample budget produced essentially noise on the degradation stability metric (Spearman near zero). Integrated Gradients is gradient-based, deterministic, ~50x faster, and well-defined for both CNN and ViT architectures. This cell rebinds the existing `shap` entry in `METHOD_REGISTRY` to use IG; the downstream Phase 3 benchmark code then works unchanged. In the dissertation, label it as "IG" in figure captions.

In [ ]:
# === V4 PATCH: Replace KernelSHAP with Integrated Gradients (Captum) ===
# KernelSHAP at n_samples=60 produced near-zero stability across degradations.
# Integrated Gradients is gradient-based — no sampling noise, ~50x faster,
# defined identically for CNN and ViT backbones.
from captum.attr import IntegratedGradients

# Define METHOD_REGISTRY here so it's available.
METHOD_REGISTRY = {
    'gradcam':   {'fn': gradcam_heatmap,    'models': ('resnet50', 'efficientnet_b3')},
    'shap':      {'fn': shap_heatmap,       'models': MODEL_NAMES},
    'attention': {'fn': attention_rollout,  'models': ('vit_base',)},
}

def ig_heatmap(model, x, target_class=None, steps=50):
    """Integrated Gradients heatmap (model-agnostic). Returns (H, W) in [0, 1]."""
    if target_class is None:
        with torch.no_grad():
            target_class = int(model(x).argmax(1).item())
    ig = IntegratedGradients(model)
    attr = ig.attribute(x, target=target_class, n_steps=steps, internal_batch_size=8)
    heat = attr.abs().sum(dim=1).squeeze().detach().cpu().numpy()
    if heat.max() > heat.min():
        heat = (heat - heat.min()) / (heat.max() - heat.min())
    return heat


# Swap into the existing registry — keep the dict key "shap" so downstream plots
# and aggregation code continue to work without edits. In the dissertation,
# rename it to "IG" in figure captions.
METHOD_REGISTRY["shap"]["fn"] = ig_heatmap
print("XAI registry updated: 'shap' entry now uses Integrated Gradients.")
print("Method dispatch:", {k: v["fn"].__name__ for k, v in METHOD_REGISTRY.items()})


In [ ]:
print(METHOD_REGISTRY['shap']['fn'].__name__)   # must print 'ig_heatmap'

## V2 — Step 2 — SHAP via GradientExplainer across CNNs + ViT

Professor deliverable #1. The original V4 pipeline replaced KernelSHAP with Integrated Gradients
because KernelSHAP took 30+ seconds per image at 384 px. `shap.GradientExplainer` is
GPU-accelerated and runs in ~3–5 s per image at 384 px — fast enough to benchmark on all three
models. The cells below:

1. Implement `shap_heatmap_grad()` and a cached background fetcher (~50 samples in GPU memory).
2. Rebind the registry: `gradcam` (CNNs), `shap` (real SHAP, all models), `ig` (renamed from
   the old `shap` IG entry), `attention` (ViT).
3. Run the Phase 3 benchmark loop with the expanded registry.
4. Add SHAP-specific quality metrics — faithfulness, sufficiency, cross-model consistency,
   SHAP-vs-Grad-CAM agreement.
5. Render the side-by-side comparison grid (clean + 3 degradation-high conditions per model).


In [ ]:
# [RESUME] SHAP (GradientExplainer) setup already done -- skipped.
print('[resume] skipped SHAP setup.')


In [ ]:
# [RESUME] SHAP quality metrics already done -- skipped.
print('[resume] skipped SHAP metrics.')


In [ ]:
# [RESUME] SHAP comparison visualisation already done -- skipped.
print('[resume] skipped SHAP comparison plots.')


In [ ]:
# [RESUME] Phase-3 XAI benchmark already done -- reload CSV instead of recomputing.
import pandas as pd
_xai_csv = P3 / 'metrics' / 'xai_results.csv'
if _xai_csv.exists():
    xai_df = pd.read_csv(_xai_csv)
    print(f'[resume] loaded {len(xai_df)} XAI rows from {_xai_csv} (benchmark skipped).')
else:
    print('[resume][warn] xai_results.csv not found on Drive; benchmark skipped anyway.')


In [ ]:
# [RESUME] Phase-3 XAI aggregated tables/plots already done -- skipped.
print('[resume] skipped XAI aggregation plots.')


In [ ]:
# ---- 1. Patch attention_rollout to find Transformer blocks inside MultiScaleClassifier ----
def _get_vit_blocks(model):
    """Locate the list of Transformer blocks regardless of wrapping depth."""
    if hasattr(model, 'blocks'):                              # raw timm ViT (v1/v2)
        return model.blocks
    bb = getattr(model, 'backbone', None)                     # v3 MultiScaleClassifier
    if bb is not None:
        if hasattr(bb, 'blocks'):
            return bb.blocks
        for attr in ('model', '_model'):
            inner = getattr(bb, attr, None)
            if inner is not None and hasattr(inner, 'blocks'):
                return inner.blocks
    # last-resort module walk
    for m in model.modules():
        if m is model: continue
        if hasattr(m, 'blocks'):
            try:
                if hasattr(m.blocks[0], 'attn'):
                    return m.blocks
            except (IndexError, TypeError):
                pass
    raise AttributeError(f"Could not find Transformer blocks in {type(model).__name__}")


@torch.no_grad()
def attention_rollout(model, x, target_class=None):
    blocks = _get_vit_blocks(model)
    attns, handles = [], []
    for blk in blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try:
        _ = model(x)
    finally:
        for h in handles: h.remove()
    if not attns:
        raise RuntimeError("attention_rollout: no attention captured — block hooks didn't fire")
    res = None
    for a in attns:
        a = a.mean(1) + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        res = a if res is None else a @ res
    cls = res[0, 0, 1:]
    side = int(cls.numel() ** 0.5)
    h = F.interpolate(cls.reshape(side, side)[None, None],
                       size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze().cpu().numpy()
    if h.max() > h.min():
        h = (h - h.min()) / (h.max() - h.min())
    return h


# Re-bind in the registry so downstream cells use the patched version
METHOD_REGISTRY['attention']['fn'] = attention_rollout
print("attention_rollout patched for v3 MultiScaleClassifier.")


# ---- 2. Render the qualitative figure ----
def overlay(img_pil, heat, size=384, alpha=0.45):
    arr = np.array(img_pil.resize((size, size))).astype(np.float32) / 255
    heat_resized = cv2.resize(heat, (size, size), interpolation=cv2.INTER_LINEAR)
    out = (1 - alpha) * arr + alpha * plt.get_cmap('jet')(heat_resized)[..., :3]
    return np.clip(out, 0, 1)


demo_id = EXPLAIN_IDS[0]
label   = int(labels_df[demo_id])
img_clean = Image.open(resolve_image(demo_id)).convert('RGB')
x_clean   = load_tensor(resolve_image(demo_id))
img_deg = Image.open(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}{DEG_SAVE_EXT}').convert('RGB')
x_deg   = load_tensor(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}{DEG_SAVE_EXT}')

VIS_SIZE = 384

fig, axes = plt.subplots(len(MODEL_NAMES), 4, figsize=(13, 3.2 * len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    method = 'attention' if name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_c = fn(models[name], x_clean, target_class=label)
    h_d = fn(models[name], x_deg,   target_class=label)
    axes[r, 0].imshow(img_clean.resize((VIS_SIZE, VIS_SIZE)))
    axes[r, 0].set_title(f'{name} clean' if r == 0 else 'clean')
    axes[r, 1].imshow(overlay(img_clean, h_c, VIS_SIZE)); axes[r, 1].set_title(f'{method} clean')
    axes[r, 2].imshow(img_deg.resize((VIS_SIZE, VIS_SIZE))); axes[r, 2].set_title('blur-high')
    axes[r, 3].imshow(overlay(img_deg, h_d, VIS_SIZE)); axes[r, 3].set_title(f'{method} blur-high')
    for ax in axes[r]:
        ax.axis('off')

plt.tight_layout()
out = P3 / 'samples' / f'qualitative_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

def fundus_mask(img_pil, size=384):
    arr = np.array(img_pil.resize((size, size)).convert('L'))
    return (arr > 7).astype(np.float32)

def overlay(img_pil, heat, size=384, alpha=0.45):
    arr = np.array(img_pil.resize((size, size))).astype(np.float32) / 255
    heat_resized = cv2.resize(heat, (size, size), interpolation=cv2.INTER_LINEAR)
    m = fundus_mask(img_pil, size)
    blended = (1 - alpha) * arr + alpha * plt.get_cmap('jet')(heat_resized * m)[..., :3]
    return np.clip(blended, 0, 1)

@torch.no_grad()
def attention_rollout_v3(model, x, target_class=None):
    """Attention rollout that finds ViT blocks regardless of wrapper depth."""

    # Recursively find the module that has .blocks containing attention
    def find_blocks(module):
        if hasattr(module, 'blocks') and len(list(module.blocks)) > 0:
            first = list(module.blocks)[0]
            if hasattr(first, 'attn'):
                return module.blocks
        for child in module.children():
            result = find_blocks(child)
            if result is not None:
                return result
        return None

    blocks = find_blocks(model)
    if blocks is None:
        raise RuntimeError("Could not find ViT blocks with .attn in model")

    attns, handles = [], []
    for blk in blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try:
        _ = model(x)
    finally:
        for h in handles: h.remove()

    res = None
    for a in attns:
        a = a.mean(1) + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        res = a if res is None else a @ res
    cls = res[0, 0, 1:]; side = int(cls.numel() ** 0.5)
    h = F.interpolate(cls.reshape(side, side)[None, None],
                       size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze().cpu().numpy()
    if h.max() > h.min(): h = (h - h.min()) / (h.max() - h.min())
    return h

# Update the registry to use the fixed version
METHOD_REGISTRY['attention']['fn'] = attention_rollout_v3

demo_id = EXPLAIN_IDS[0]; label = int(labels_df[demo_id])
img_clean = Image.open(resolve_image(demo_id)).convert('RGB')
x_clean = load_tensor(resolve_image(demo_id))
img_deg = Image.open(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}{DEG_SAVE_EXT}').convert('RGB')
x_deg = load_tensor(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}{DEG_SAVE_EXT}')

VIS_SIZE = 384

fig, axes = plt.subplots(len(MODEL_NAMES), 4, figsize=(13, 3.2*len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    method = 'attention' if name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_c = fn(models[name], x_clean, target_class=label)
    h_d = fn(models[name], x_deg,   target_class=label)
    axes[r, 0].imshow(img_clean.resize((VIS_SIZE, VIS_SIZE))); axes[r, 0].set_title(f'{name} clean' if r == 0 else 'clean')
    axes[r, 1].imshow(overlay(img_clean, h_c, VIS_SIZE)); axes[r, 1].set_title(f'{method} clean')
    axes[r, 2].imshow(img_deg.resize((VIS_SIZE, VIS_SIZE))); axes[r, 2].set_title('blur-high')
    axes[r, 3].imshow(overlay(img_deg, h_d, VIS_SIZE)); axes[r, 3].set_title(f'{method} blur-high')
    for ax in axes[r]: ax.axis('off')
plt.tight_layout()
out = P3 / 'samples' / f'qualitative_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

In [ ]:
demo_id = EXPLAIN_IDS[0]; label = int(labels_df[demo_id])
img_clean = Image.open(resolve_image(demo_id)).convert('RGB')
x_clean = load_tensor(resolve_image(demo_id))
VIS_SIZE = 384

# ── BLUR ──
conditions = [('clean', None, None), ('low', 'blur', 'low'), ('mid', 'blur', 'mid'), ('high', 'blur', 'high')]
metric_rows = []

fig, axes = plt.subplots(len(MODEL_NAMES), len(conditions) * 2, figsize=(6 * len(conditions), 3.2 * len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    method = 'attention' if name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_clean_ref = None
    for c, (tag, kind, level) in enumerate(conditions):
        if tag == 'clean':
            img = img_clean; x = x_clean
        else:
            path = DEGRADED_DIR / kind / level / f'{demo_id}{DEG_SAVE_EXT}'
            img = Image.open(path).convert('RGB')
            x = load_tensor(path)
        heat = fn(models[name], x, target_class=label)

        # Store clean heatmap as reference
        if tag == 'clean':
            h_clean_ref = heat

        # Compute metrics
        ins_auc = insertion_auc(models[name], x, heat, label)
        del_auc = deletion_auc(models[name], x, heat, label)
        stab = 1.0 if tag == 'clean' else stability_spearman(h_clean_ref, heat)
        metric_rows.append({'model': name, 'method': method, 'condition': tag,
                            'insertion_auc': round(ins_auc, 4),
                            'deletion_auc': round(del_auc, 4),
                            'stability': round(stab, 4)})

        col = c * 2
        axes[r, col].imshow(img.resize((VIS_SIZE, VIS_SIZE)))
        axes[r, col].set_title(f'{name} {tag}', fontsize=10)
        axes[r, col].axis('off')

        # Add metrics as subtitle on heatmap
        axes[r, col+1].imshow(overlay(img, heat, VIS_SIZE))
        axes[r, col+1].set_title(f'{method} {tag}\nIns:{ins_auc:.3f} Del:{del_auc:.3f} Stab:{stab:.3f}', fontsize=8)
        axes[r, col+1].axis('off')

fig.suptitle(f'BLUR Degradation Progression — {demo_id}', fontsize=14, fontweight='bold')
plt.tight_layout()
out = P3 / 'samples' / f'blur_progression_metrics_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

blur_metrics = pd.DataFrame(metric_rows)
print('\n=== BLUR METRICS ===')
print(blur_metrics.to_string(index=False))
blur_metrics.to_csv(P3 / 'metrics' / f'blur_progression_metrics_{demo_id}.csv', index=False)

In [ ]:
# ── EXPOSURE ──
conditions = [('clean', None, None), ('low', 'exposure', 'low'), ('mid', 'exposure', 'mid'), ('high', 'exposure', 'high')]
metric_rows = []

fig, axes = plt.subplots(len(MODEL_NAMES), len(conditions) * 2, figsize=(6 * len(conditions), 3.2 * len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    method = 'attention' if name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_clean_ref = None
    for c, (tag, kind, level) in enumerate(conditions):
        if tag == 'clean':
            img = img_clean; x = x_clean
        else:
            path = DEGRADED_DIR / kind / level / f'{demo_id}{DEG_SAVE_EXT}'
            img = Image.open(path).convert('RGB')
            x = load_tensor(path)
        heat = fn(models[name], x, target_class=label)
        if tag == 'clean': h_clean_ref = heat

        ins_auc = insertion_auc(models[name], x, heat, label)
        del_auc = deletion_auc(models[name], x, heat, label)
        stab = 1.0 if tag == 'clean' else stability_spearman(h_clean_ref, heat)
        metric_rows.append({'model': name, 'method': method, 'condition': tag,
                            'insertion_auc': round(ins_auc, 4),
                            'deletion_auc': round(del_auc, 4),
                            'stability': round(stab, 4)})

        col = c * 2
        axes[r, col].imshow(img.resize((VIS_SIZE, VIS_SIZE)))
        axes[r, col].set_title(f'{name} {tag}', fontsize=10)
        axes[r, col].axis('off')
        axes[r, col+1].imshow(overlay(img, heat, VIS_SIZE))
        axes[r, col+1].set_title(f'{method} {tag}\nIns:{ins_auc:.3f} Del:{del_auc:.3f} Stab:{stab:.3f}', fontsize=8)
        axes[r, col+1].axis('off')

fig.suptitle(f'EXPOSURE Degradation Progression — {demo_id}', fontsize=14, fontweight='bold')
plt.tight_layout()
out = P3 / 'samples' / f'exposure_progression_metrics_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

exposure_metrics = pd.DataFrame(metric_rows)
print('\n=== EXPOSURE METRICS ===')
print(exposure_metrics.to_string(index=False))
exposure_metrics.to_csv(P3 / 'metrics' / f'exposure_progression_metrics_{demo_id}.csv', index=False)

In [ ]:
# ── NOISE ──
conditions = [('clean', None, None), ('low', 'noise', 'low'), ('mid', 'noise', 'mid'), ('high', 'noise', 'high')]
metric_rows = []

fig, axes = plt.subplots(len(MODEL_NAMES), len(conditions) * 2, figsize=(6 * len(conditions), 3.2 * len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    method = 'attention' if name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_clean_ref = None
    for c, (tag, kind, level) in enumerate(conditions):
        if tag == 'clean':
            img = img_clean; x = x_clean
        else:
            path = DEGRADED_DIR / kind / level / f'{demo_id}{DEG_SAVE_EXT}'
            img = Image.open(path).convert('RGB')
            x = load_tensor(path)
        heat = fn(models[name], x, target_class=label)
        if tag == 'clean': h_clean_ref = heat

        ins_auc = insertion_auc(models[name], x, heat, label)
        del_auc = deletion_auc(models[name], x, heat, label)
        stab = 1.0 if tag == 'clean' else stability_spearman(h_clean_ref, heat)
        metric_rows.append({'model': name, 'method': method, 'condition': tag,
                            'insertion_auc': round(ins_auc, 4),
                            'deletion_auc': round(del_auc, 4),
                            'stability': round(stab, 4)})

        col = c * 2
        axes[r, col].imshow(img.resize((VIS_SIZE, VIS_SIZE)))
        axes[r, col].set_title(f'{name} {tag}', fontsize=10)
        axes[r, col].axis('off')
        axes[r, col+1].imshow(overlay(img, heat, VIS_SIZE))
        axes[r, col+1].set_title(f'{method} {tag}\nIns:{ins_auc:.3f} Del:{del_auc:.3f} Stab:{stab:.3f}', fontsize=8)
        axes[r, col+1].axis('off')

fig.suptitle(f'NOISE Degradation Progression — {demo_id}', fontsize=14, fontweight='bold')
plt.tight_layout()
out = P3 / 'samples' / f'noise_progression_metrics_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

noise_metrics = pd.DataFrame(metric_rows)
print('\n=== NOISE METRICS ===')
print(noise_metrics.to_string(index=False))
noise_metrics.to_csv(P3 / 'metrics' / f'noise_progression_metrics_{demo_id}.csv', index=False)

In [ ]:
# Qualitative IG figure — Integrated Gradients heatmap for ALL three models,
# clean vs degraded, side-by-side. Complements the GradCAM/Attention figure.

ig_fn = METHOD_REGISTRY['shap']['fn']      # this is ig_heatmap after the V4 IG swap
print('IG function in registry :', ig_fn.__name__)

demo_id  = EXPLAIN_IDS[0]
label    = int(labels_df[demo_id])

img_clean = Image.open(resolve_image(demo_id)).convert('RGB')
x_clean   = load_tensor(resolve_image(demo_id))
img_deg   = Image.open(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}{DEG_SAVE_EXT}').convert('RGB')
x_deg     = load_tensor(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}{DEG_SAVE_EXT}')

VIS_SIZE = 384

fig, axes = plt.subplots(len(MODEL_NAMES), 4, figsize=(13, 3.2 * len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    print(f'  computing IG for {name}...')
    h_c = ig_fn(models[name], x_clean, target_class=label)
    h_d = ig_fn(models[name], x_deg,   target_class=label)
    axes[r, 0].imshow(img_clean.resize((VIS_SIZE, VIS_SIZE)))
    axes[r, 0].set_title(f'{name} clean' if r == 0 else 'clean')
    axes[r, 1].imshow(overlay(img_clean, h_c, VIS_SIZE))
    axes[r, 1].set_title('IG clean')
    axes[r, 2].imshow(img_deg.resize((VIS_SIZE, VIS_SIZE)))
    axes[r, 2].set_title('blur-high')
    axes[r, 3].imshow(overlay(img_deg, h_d, VIS_SIZE))
    axes[r, 3].set_title('IG blur-high')
    for ax in axes[r]:
        ax.axis('off')

fig.suptitle('Integrated Gradients (model-agnostic explanation) — clean vs blur-high',
              fontsize=13, y=1.005)
plt.tight_layout()
out = P3 / 'samples' / f'qualitative_IG_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

In [ ]:
# === V2 PATCH: relabel cell neutralised ===
# In V1 this cell renamed every 'shap' row in xai_results.csv to 'IG' because
# the V4 patch had repurposed the registry key 'shap' to point at IG. In V2
# the registry has BOTH 'shap' (real SHAP via GradientExplainer) and 'ig'
# (renamed from the legacy 'shap'). Renaming would now corrupt the labels.
# We leave the CSV untouched.
import pandas as pd
xai_csv = P3 / 'metrics' / 'xai_results.csv'
if xai_csv.exists():
    df = pd.read_csv(xai_csv)
    print('Method label counts in xai_results.csv (untouched):')
    print(df['method'].value_counts().to_string())
else:
    print('[skip] xai_results.csv not yet written')


In [ ]:
# [RESUME] SHAP/IG degradation-progression figures already generated -- skipped.
# Preserve the only thing the rest of the notebook reads from this cell:
N_FIGURES = 3
SAMPLE_IDS = EXPLAIN_IDS[:N_FIGURES]
print(f'[resume] skipped IG progression; SAMPLE_IDS={list(SAMPLE_IDS)}')


In [ ]:
P4 = PHASE_DIRS['phase4_genai_enhancement']

# 4.1 CLAHE baseline
_clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
def enhance_clahe(img):
    arr = np.array(img.convert('RGB'))
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    lab[..., 0] = _clahe.apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

---

# Phase 4 — GenAI Image Restoration (RQ3)

**Literature-review framing.** Three families of fundus-image restoration are evaluated, each representing a different point on the complexity vs flexibility trade-off:

| Method | Year | Class | What it learns | Why we benchmark it |
|---|---|---|---|---|
| **CLAHE** | 1994 | Classical (parameter-free) | Histogram equalisation per tile | Mandatory baseline — every clinical pipeline that pre-processes fundus images uses it. |
| **Real-ESRGAN** | 2021 | Generative (RRDBNet generator, no attention) | Real-world blind super-resolution with synthetic degradation modelling | Robust, off-the-shelf; the de-facto open-source restoration model. |
| **A-ESRGAN** | 2022 | Generative (RRDBNet + attention-U-Net discriminator) | Same as Real-ESRGAN at inference, but trained against an attention-U-Net discriminator that focuses the adversarial signal on regions of fine detail | Hypothesised better lesion / vessel preservation thanks to the attention discriminator's training-time emphasis on locally-detailed regions. |

The **inference-time generator architecture is identical** between Real-ESRGAN and A-ESRGAN (RRDBNet, 23 residual-in-residual dense blocks). The performance difference, if any, is attributable purely to **training signal** — not parameter count or capacity. This makes for a clean controlled comparison in the dissertation.

**Evaluation protocol.** For each (degradation type, severity) split we run:
1. **Raw** — degraded image straight into the classifier (the v3 stress-test result, our reference).
2. **CLAHE** — classical baseline.
3. **Real-ESRGAN** or **A-ESRGAN** — GenAI restoration.

Outputs: accuracy / F1 / kappa per (model, degradation, level, variant), saved to `results/phase4_genai_enhancement/metrics/recovery_accuracy.csv`. Recovery is plotted with the v2 stress-test as the lower reference and clean-image accuracy as the upper reference.


In [ ]:
# 4.2 GenAI restoration — A-ESRGAN (primary) -> Real-ESRGAN -> TinyU-Net
#
# A-ESRGAN (Wei et al., 2022) keeps Real-ESRGAN's RRDBNet generator and adds
# attention-U-Net discriminators during training. At INFERENCE time only the
# generator is used, so we load A-ESRGAN's authors' pretrained generator
# weights into the same RRDBNet architecture used by Real-ESRGAN. If those
# weights cannot be downloaded, we fall back to standard Real-ESRGAN x2,
# then to a TinyU-Net trained on the pristine set.

%pip install -q basicsr realesrgan facexlib gfpgan

import urllib.request, os

def _download(url: str, dest: Path) -> bool:
    if dest.exists() and dest.stat().st_size > 0:
        return True
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        print(f"  downloading {url} -> {dest.name}")
        urllib.request.urlretrieve(url, dest)
        return dest.exists() and dest.stat().st_size > 0
    except Exception as e:
        print(f"  download failed: {e}")
        return False


def _try_aesrgan():
    """A-ESRGAN x2 — RRDBNet generator + A-ESRGAN-trained weights.

    Reference: Wei et al. (2022) "A-ESRGAN: Training Real-World Blind
    Super-Resolution with Attention U-Net Discriminators".
    """
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        # Authors' release URL (mirror via HuggingFace if GitHub asset is rate-limited).
        url   = "https://huggingface.co/aesrgan/A-ESRGAN-Single/resolve/main/A_ESRGAN_Single.pth"
        ckpt  = CHECKPOINTS_DIR / "A_ESRGAN_Single.pth"
        if not _download(url, ckpt):
            return None
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                      num_block=23, num_grow_ch=32, scale=4)
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        # A-ESRGAN saves under 'params' or 'params_ema'
        sd = state.get("params_ema", state.get("params", state))
        net.load_state_dict(sd, strict=False)
        net = net.to(DEVICE).eval()

        @torch.no_grad()
        def process(img_pil):
            arr = np.asarray(img_pil.convert("RGB"), dtype=np.float32) / 255.0
            x   = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            with autocast(enabled=True):
                y = net(x).clamp(0, 1)
            out = (y.squeeze().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
            return Image.fromarray(out).resize(img_pil.size)
        return process, "a_esrgan_x4"
    except Exception as e:
        print(f"[a_esrgan failed] {e}")
        return None


def _try_realesrgan():
    """Real-ESRGAN x2 — robust off-the-shelf restoration."""
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        url  = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"
        ckpt = CHECKPOINTS_DIR / "RealESRGAN_x2plus.pth"
        if not _download(url, ckpt):
            return None
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                      num_block=23, num_grow_ch=32, scale=2)
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        sd = state.get("params_ema", state.get("params", state))
        net.load_state_dict(sd, strict=False)
        net = net.to(DEVICE).eval()

        @torch.no_grad()
        def process(img_pil):
            arr = np.asarray(img_pil.convert("RGB"), dtype=np.float32) / 255.0
            x   = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            with autocast(enabled=True):
                y = net(x).clamp(0, 1)
            out = (y.squeeze().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
            return Image.fromarray(out).resize(img_pil.size)
        return process, "real_esrgan_x2"
    except Exception as e:
        print(f"[real_esrgan failed] {e}")
        return None


# Tiny U-Net retained as a last-resort fallback so Phase 4 always runs.
class TinyUNet(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        def conv(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(True),
                                 nn.Conv2d(o, o, 3, padding=1), nn.ReLU(True))
        self.e1 = conv(3, ch); self.e2 = conv(ch, ch*2); self.e3 = conv(ch*2, ch*4)
        self.up2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2); self.d2 = conv(ch*4, ch*2)
        self.up1 = nn.ConvTranspose2d(ch*2, ch, 2, stride=2);   self.d1 = conv(ch*2, ch)
        self.out = nn.Conv2d(ch, 3, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2))
        d2 = self.d2(torch.cat([self.up2(e3), e2], 1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))


def _train_tiny_unet(epochs=2, n=400):
    df = pd.read_csv(PRISTINE_CSV).sample(n, random_state=0)
    tfm = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
    mdl = TinyUNet().to(DEVICE)
    optim = torch.optim.AdamW(mdl.parameters(), lr=1e-3); crit = nn.L1Loss()
    for _ in range(epochs):
        for row in df.sample(frac=1).itertuples():
            try:
                img = Image.open(resolve_image(row.id_code)).convert("RGB")
            except FileNotFoundError:
                continue
            target = tfm(img).unsqueeze(0).to(DEVICE)
            if np.random.rand() < 0.5:
                noisy = (target + 0.1 * torch.randn_like(target)).clamp(0, 1)
            else:
                k = 5; w = torch.ones((3, 1, k, k), device=DEVICE) / (k * k)
                noisy = F.conv2d(target, w, padding=k // 2, groups=3)
            optim.zero_grad(); crit(mdl(noisy), target).backward(); optim.step()
    return mdl


def _make_unet_processor(mdl):
    tfm = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
    @torch.no_grad()
    def process(img_pil):
        x = tfm(img_pil).unsqueeze(0).to(DEVICE)
        out = mdl(x).squeeze().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        return Image.fromarray((out * 255).astype(np.uint8)).resize(img_pil.size)
    return process


# ---- Cascade: try the best option that loads successfully ----
result = _try_aesrgan() or _try_realesrgan()
if result is not None:
    enhance_genai, GENAI_BACKEND = result
else:
    print("Falling back to TinyU-Net (training ~1 min)...")
    ckpt = CHECKPOINTS_DIR / "genai_unet.pt"
    unet = TinyUNet().to(DEVICE)
    if ckpt.exists():
        unet.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
    else:
        unet = _train_tiny_unet()
        torch.save(unet.state_dict(), ckpt)
    enhance_genai = _make_unet_processor(unet)
    GENAI_BACKEND = "tiny_unet"

print(f"GenAI backend: {GENAI_BACKEND}")


## V5 - Cold Diffusion for fundus restoration (replaces SD img2img)

The earlier Stable-Diffusion-2.1 img2img variant was **removed**: SD was trained on LAION, not fundus, and even at `strength=0.30` it hallucinates anatomically-plausible-but-wrong vessels and can fabricate lesions the classifier then "detects".

This cell replaces it with **Cold Diffusion** (Bansal et al., 2022). Cold Diffusion swaps the Gaussian-noise forward process for an arbitrary deterministic degradation operator — and you happen to already have those operators in Phase 1 (`gaussian_blur`, `exposure_shift`, `gaussian_noise`). So the model is trained to invert your *exact* synthetic pipeline. Lesion-preserving by construction, fast (~90 min train, 8-step inference), and a clean novel-contribution story for the dissertation.


In [ ]:
# === V5 / V2 PATCH: Cold Diffusion for fundus restoration ===
# Bansal et al. 2022 (https://arxiv.org/abs/2208.09392): the diffusion forward
# process can be any deterministic degradation D(x, t). Train R_theta so that
# R(D(x, t), t) ≈ x for all t, then sample via Algorithm 2:
#
#       x_{s-1} = x_s - D(R(x_s, s), s) + D(R(x_s, s), s-1)
#
# V2 patch (Step 1c) fixes the noise-catastrophe documented in OBSERVATIONS:
#   - 15 epochs with CosineAnnealingLR (was 4 epochs, constant LR)
#   - Kind sampling biased to noise (was uniform)
#   - New checkpoint name cold_diffusion_v6.pt so the resume guard fires fresh

import math, random

COLD_DIFF_CKPT = CHECKPOINTS_DIR / 'cold_diffusion_v6.pt'   # V2 patch: bumped from v5
COLD_DIFF_SIZE = 256        # work at 256 px to fit budget; resize back at end
T_STEPS        = 8          # train + inference steps

# Severity schedule: t in [0, T_STEPS] -> degradation parameter
SEV_HIGH = {'blur': 9.0, 'exposure_gain_low': 0.2, 'noise': 0.12}

def _severity(t, kind):
    f = t / T_STEPS  # in [0, 1]
    if kind == 'blur':
        return max(0.01, f * SEV_HIGH['blur'])
    if kind == 'exposure':
        return 1.0 - f * (1.0 - SEV_HIGH['exposure_gain_low'])
    if kind == 'noise':
        return f * SEV_HIGH['noise']
    raise ValueError(kind)

def _degrade_t(img_pil, kind, t):
    """Forward operator at severity t using the Phase-1 functions."""
    if t <= 0:
        return img_pil
    p = _severity(t, kind)
    return DEGRADERS[kind](img_pil, p)


# ---- Conditional U-Net (FiLM on kind + t/T) ----
class CondUNet(nn.Module):
    def __init__(self, ch=48, n_kinds=3, t_emb=64):
        super().__init__()
        self.kind_emb = nn.Embedding(n_kinds, t_emb)
        self.t_proj   = nn.Sequential(
            nn.Linear(1, t_emb), nn.SiLU(), nn.Linear(t_emb, t_emb),
        )
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.GroupNorm(8, o), nn.SiLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.GroupNorm(8, o), nn.SiLU(),
            )
        self.in_proj = nn.Conv2d(3, ch, 3, padding=1)
        self.e1 = block(ch,   ch)
        self.e2 = block(ch,   ch*2)
        self.e3 = block(ch*2, ch*4)
        self.mid = block(ch*4, ch*4)
        self.film = nn.Linear(t_emb*2, ch*4*2)
        self.up2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2); self.d2 = block(ch*4, ch*2)
        self.up1 = nn.ConvTranspose2d(ch*2, ch,   2, stride=2); self.d1 = block(ch*2, ch)
        self.out = nn.Conv2d(ch, 3, 1)
    def forward(self, x, kind_id, t_norm):
        cond = torch.cat([self.kind_emb(kind_id),
                          self.t_proj(t_norm.unsqueeze(-1))], dim=-1)
        gb = self.film(cond)
        g, b = gb.chunk(2, dim=-1)
        h0 = self.in_proj(x)
        e1 = self.e1(h0)
        e2 = self.e2(F.avg_pool2d(e1, 2))
        e3 = self.e3(F.avg_pool2d(e2, 2))
        m  = self.mid(e3)
        m  = m * (1 + g[:, :, None, None]) + b[:, :, None, None]
        d2 = self.d2(torch.cat([self.up2(m),  e2], 1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))


KIND_TO_ID = {'blur': 0, 'exposure': 1, 'noise': 2}
# V2 patch (Step 1c): weight noise samples higher in training to fix the noise catastrophe
KIND_SAMPLE_WEIGHTS = {'blur': 0.25, 'exposure': 0.25, 'noise': 0.50}

class _ColdDiffDataset(Dataset):
    """On-the-fly paired (degraded@t, clean) sampler over the pristine set.

    V2 patch: kind sampling uses KIND_SAMPLE_WEIGHTS so noise dominates training.
    """
    def __init__(self, df, n_per=4):
        self.df = df.reset_index(drop=True); self.n_per = n_per
        self.tfm = T.Compose([T.Resize((COLD_DIFF_SIZE, COLD_DIFF_SIZE)),
                              T.ToTensor()])
        self._kinds = list(KIND_SAMPLE_WEIGHTS.keys())
        self._probs = list(KIND_SAMPLE_WEIGHTS.values())
    def __len__(self):
        return len(self.df) * self.n_per
    def __getitem__(self, i):
        row = self.df.iloc[i // self.n_per]
        try:
            img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (COLD_DIFF_SIZE, COLD_DIFF_SIZE))
        img  = img.resize((COLD_DIFF_SIZE, COLD_DIFF_SIZE), Image.BILINEAR)
        kind = random.choices(self._kinds, weights=self._probs, k=1)[0]
        t    = random.randint(1, T_STEPS)
        deg  = _degrade_t(img, kind, t)
        return (self.tfm(deg), KIND_TO_ID[kind],
                torch.tensor(t / T_STEPS, dtype=torch.float32),
                self.tfm(img))


def train_cold_diffusion(epochs=15, batch_size=16, lr=2e-4):
    """V2 patch: 15 epochs, CosineAnnealingLR, noise-biased sampling.

    Estimated time: ~5 hours on A100 with T_STEPS=8, COLD_DIFF_SIZE=256.
    """
    if COLD_DIFF_CKPT.exists():
        print(f"[skip] {COLD_DIFF_CKPT.name} already on Drive — not retraining.")
        return
    print(f"Training Cold Diffusion v6 ({epochs} epochs, cosine LR, noise-biased)...")
    df  = pd.read_csv(PRISTINE_CSV)
    ds  = _ColdDiffDataset(df, n_per=4)
    dl  = DataLoader(ds, batch_size=batch_size, shuffle=True,
                     num_workers=2, pin_memory=True)
    net = CondUNet().to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = GradScaler(enabled=True)
    crit   = nn.L1Loss()
    history = []
    for ep in range(epochs):
        net.train()
        running, n = 0.0, 0
        pbar = tqdm(dl, desc=f"cold-diff v6 ep {ep+1}/{epochs}", leave=False)
        for deg, kind_id, tnorm, clean in pbar:
            deg   = deg.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            kind_id = kind_id.to(DEVICE).long()
            tnorm   = tnorm.to(DEVICE).float()
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=True):
                pred = net(deg, kind_id, tnorm)
                loss = crit(pred, clean)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            running += loss.item() * deg.size(0); n += deg.size(0)
            pbar.set_postfix(L1=f"{running/n:.4f}",
                             lr=f"{opt.param_groups[0]['lr']:.2e}")
        sched.step()
        epoch_l1 = running / max(n, 1)
        history.append({'epoch': ep+1, 'l1': epoch_l1, 'lr': opt.param_groups[0]['lr']})
        print(f"  ep {ep+1}: L1={epoch_l1:.4f}  lr={opt.param_groups[0]['lr']:.2e}")
    torch.save({'state_dict': net.state_dict(),
                'T_STEPS': T_STEPS,
                'kind_weights': KIND_SAMPLE_WEIGHTS,
                'history': history}, COLD_DIFF_CKPT)
    print(f"Saved -> {COLD_DIFF_CKPT}")


train_cold_diffusion(epochs=15, batch_size=16, lr=2e-4)


# ---- Inference: Cold Diffusion Algorithm 2 sampling ----
_cold_net = None
def _load_cold_net():
    global _cold_net
    if _cold_net is None:
        _cold_net = CondUNet().to(DEVICE).eval()
        sd = torch.load(COLD_DIFF_CKPT, map_location=DEVICE, weights_only=False)
        _cold_net.load_state_dict(sd['state_dict'])
    return _cold_net


@torch.no_grad()
def enhance_cold_diffusion(img_pil, kind='blur', t_start=T_STEPS):
    """Iterative restoration via Cold Diffusion Algorithm 2."""
    net = _load_cold_net()
    orig_size = img_pil.size
    src = img_pil.convert('RGB').resize((COLD_DIFF_SIZE, COLD_DIFF_SIZE), Image.BILINEAR)
    x = T.ToTensor()(src).unsqueeze(0).to(DEVICE)
    kind_id = torch.tensor([KIND_TO_ID[kind]], device=DEVICE, dtype=torch.long)
    to_pil = T.ToPILImage()
    to_t   = T.ToTensor()
    for s in range(t_start, 0, -1):
        tnorm = torch.tensor([s / T_STEPS], device=DEVICE)
        x_hat = net(x, kind_id, tnorm).clamp(0, 1)
        if s > 1:
            xh_pil = to_pil(x_hat.squeeze().cpu())
            d_s   = _degrade_t(xh_pil, kind, s)
            d_sm1 = _degrade_t(xh_pil, kind, s - 1)
            d_s   = to_t(d_s.resize((COLD_DIFF_SIZE, COLD_DIFF_SIZE))).unsqueeze(0).to(DEVICE)
            d_sm1 = to_t(d_sm1.resize((COLD_DIFF_SIZE, COLD_DIFF_SIZE))).unsqueeze(0).to(DEVICE)
            x = (x - d_s + d_sm1).clamp(0, 1)
        else:
            x = x_hat
    arr = (x.squeeze().permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype("uint8")
    return Image.fromarray(arr).resize(orig_size)


COLD_T_FOR_LEVEL = {'low': 3, 'mid': 5, 'high': T_STEPS}
print("Cold Diffusion v6 restorer ready.")


In [ ]:
  from PIL import Image
  import matplotlib.pyplot as plt

  test_id = list(test_id_set)[0]
  clean = Image.open(resolve_image(test_id)).convert('RGB').resize((256, 256))
  deg   = _degrade_t(clean, 'blur', T_STEPS)         # max severity
  rest  = enhance_cold_diffusion(deg, kind='blur', t_start=T_STEPS)

  fig, ax = plt.subplots(1, 3, figsize=(12, 4))
  for a, im, t in zip(ax, [clean, deg, rest], ['clean', 'blur-high', 'restored']):
      a.imshow(im); a.set_title(t); a.axis('off')
  plt.show()

## V5 - SwinIR + GAN restorer (4th enhancer)

A second supervised restorer alongside Cold Diffusion. SwinIR (Liang et al., 2021) is a Swin-Transformer backbone purpose-built for image restoration; pairing it with a PatchGAN discriminator gives the perceptual sharpness that pure L1 lacks while still being supervised on paired (clean, degraded) data.

- **Architecture**: slim SwinIR (embed_dim=60, depths=[2,2,2,2], window=8, ~1M params). Tries `basicsr.archs.swinir_arch.SwinIR` first; falls back to a built-in slim Swin-UNet if basicsr's version is unavailable.
- **Loss**: `L1(R(deg), clean) + 0.01 * BCE(D(deg, R(deg)), 1)` with conditional PatchGAN.
- **Training**: 3 epochs, batch 4, on-the-fly Phase-1 degradation. ~90 min on A100, ~3 hr on T4. Resume-guarded by `swinir_gan_v5.pt` on Drive.
- **Inference**: single forward pass, ~0.1 s/img — much faster than Cold Diffusion's 8-step iteration.


In [ ]:
# === V5 / V2 PATCH: SwinIR + GAN restorer ===
# Supervised paired restoration with adversarial term.
# Forward operator: the SAME Phase-1 degradation primitives, like Cold Diffusion.
#
# V2 patch (Step 1e) fixes the adversarial-drift documented in OBSERVATIONS:
#   - 8 epochs (was 3) so the generator has time to keep pace
#   - adv_w lowered 0.01 -> 0.005 so reconstruction loss dominates early
#   - Discriminator frozen after epoch 4 (freeze_d_after) to stop D-drift
#   - New checkpoint name swinir_gan_v6.pt so the resume guard fires fresh

SWINIR_CKPT = CHECKPOINTS_DIR / 'swinir_gan_v6.pt'   # V2 patch: bumped from v5
SWINIR_SIZE = 256
SWINIR_WIN  = 8

# ---- Generator: try basicsr SwinIR, fall back to slim Swin-UNet ----
def _make_swinir():
    try:
        from basicsr.archs.swinir_arch import SwinIR
        net = SwinIR(
            upscale=1, in_chans=3, img_size=SWINIR_SIZE, window_size=SWINIR_WIN,
            depths=[2, 2, 2, 2], embed_dim=60, num_heads=[2, 2, 2, 2],
            mlp_ratio=2.0, upsampler='', resi_connection='1conv',
        )
        print("  generator: basicsr SwinIR (slim)")
        return net
    except Exception as e:
        print(f"  [fallback] basicsr SwinIR unavailable ({e}); using slim Swin-UNet")
    class _Block(nn.Module):
        def __init__(self, ch):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv2d(ch, ch, 3, padding=1), nn.GELU(),
                nn.Conv2d(ch, ch, 3, padding=1),
            )
        def forward(self, x): return x + self.conv(x)
    class _SwinUNetSlim(nn.Module):
        def __init__(self, ch=48):
            super().__init__()
            self.in_proj = nn.Conv2d(3, ch, 3, padding=1)
            self.e1 = nn.Sequential(_Block(ch), _Block(ch))
            self.d1 = nn.Conv2d(ch, ch*2, 4, stride=2, padding=1)
            self.e2 = nn.Sequential(_Block(ch*2), _Block(ch*2))
            self.d2 = nn.Conv2d(ch*2, ch*4, 4, stride=2, padding=1)
            self.mid = nn.Sequential(_Block(ch*4), _Block(ch*4))
            self.u2  = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2)
            self.dec2 = nn.Sequential(_Block(ch*2), _Block(ch*2))
            self.u1  = nn.ConvTranspose2d(ch*2, ch, 2, stride=2)
            self.dec1 = nn.Sequential(_Block(ch), _Block(ch))
            self.out  = nn.Conv2d(ch, 3, 3, padding=1)
        def forward(self, x):
            h0 = self.in_proj(x)
            e1 = self.e1(h0)
            e2 = self.e2(self.d1(e1))
            m  = self.mid(self.d2(e2))
            u2 = self.dec2(self.u2(m) + e2)
            u1 = self.dec1(self.u1(u2) + e1)
            return torch.sigmoid(self.out(u1) + x)
    return _SwinUNetSlim()


# ---- Conditional PatchGAN discriminator ----
class _PatchGAN(nn.Module):
    def __init__(self, in_ch=6, base=48):
        super().__init__()
        def blk(i, o, s, norm=True):
            l = [nn.Conv2d(i, o, 4, stride=s, padding=1)]
            if norm:
                l.append(nn.GroupNorm(8, o))
            l.append(nn.LeakyReLU(0.2, True))
            return l
        self.net = nn.Sequential(
            *blk(in_ch, base,   2, norm=False),
            *blk(base,  base*2, 2),
            *blk(base*2,base*4, 2),
            *blk(base*4,base*8, 1),
            nn.Conv2d(base*8, 1, 4, padding=1),
        )
    def forward(self, x): return self.net(x)


# ---- Training dataset: same Phase-1 operators ----
class _SwinIRDataset(Dataset):
    def __init__(self, df, n_per=4):
        self.df = df.reset_index(drop=True); self.n_per = n_per
        self.tfm = T.Compose([T.Resize((SWINIR_SIZE, SWINIR_SIZE)), T.ToTensor()])
    def __len__(self):
        return len(self.df) * self.n_per
    def __getitem__(self, i):
        row = self.df.iloc[i // self.n_per]
        try:
            img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (SWINIR_SIZE, SWINIR_SIZE))
        img = img.resize((SWINIR_SIZE, SWINIR_SIZE), Image.BILINEAR)
        kind  = random.choice(['blur', 'exposure', 'noise'])
        level = random.choice(['low', 'mid', 'high'])
        deg   = DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])
        return self.tfm(deg), self.tfm(img)


def train_swinir_gan(epochs=8, batch_size=4, lr_g=1e-4, lr_d=1e-4,
                     adv_w=0.005, freeze_d_after=4):
    """V2 patch (Step 1e): 8 epochs, adv_w=0.005, freeze D after epoch 4."""
    if SWINIR_CKPT.exists():
        print(f"[skip] {SWINIR_CKPT.name} already on Drive — not retraining.")
        return
    print(f"Training SwinIR + GAN v6 ({epochs} epochs, adv_w={adv_w}, freeze D after ep {freeze_d_after})...")
    df = pd.read_csv(PRISTINE_CSV)
    ds = _SwinIRDataset(df, n_per=4)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True,
                    num_workers=2, pin_memory=True)
    G = _make_swinir().to(DEVICE)
    D = _PatchGAN(in_ch=6).to(DEVICE)
    opt_g = torch.optim.AdamW(G.parameters(), lr=lr_g, betas=(0.9, 0.99), weight_decay=1e-5)
    opt_d = torch.optim.AdamW(D.parameters(), lr=lr_d, betas=(0.9, 0.99), weight_decay=1e-5)
    scaler_g = GradScaler(enabled=True)
    scaler_d = GradScaler(enabled=True)
    l1  = nn.L1Loss()
    bce = nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        G.train()
        train_d_this_epoch = ep < freeze_d_after
        if train_d_this_epoch:
            D.train()
            for p in D.parameters(): p.requires_grad = True
        else:
            D.eval()
            for p in D.parameters(): p.requires_grad = False
        run_g, run_d, n = 0.0, 0.0, 0
        pbar = tqdm(dl, desc=f"swinir-gan v6 ep {ep+1}/{epochs}", leave=False)
        for deg, clean in pbar:
            deg   = deg.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            # ----- D step (only while not frozen) -----
            d_loss_v = 0.0
            if train_d_this_epoch:
                opt_d.zero_grad(set_to_none=True)
                with autocast(enabled=True):
                    with torch.no_grad():
                        fake = G(deg).clamp(0, 1)
                    d_real = D(torch.cat([deg, clean], 1))
                    d_fake = D(torch.cat([deg, fake], 1))
                    d_loss = 0.5 * (bce(d_real, torch.ones_like(d_real)) +
                                    bce(d_fake, torch.zeros_like(d_fake)))
                scaler_d.scale(d_loss).backward()
                scaler_d.step(opt_d); scaler_d.update()
                d_loss_v = d_loss.item()
            # ----- G step -----
            opt_g.zero_grad(set_to_none=True)
            with autocast(enabled=True):
                fake = G(deg).clamp(0, 1)
                d_out = D(torch.cat([deg, fake], 1))
                adv  = bce(d_out, torch.ones_like(d_out))
                rec  = l1(fake, clean)
                g_loss = rec + adv_w * adv
            scaler_g.scale(g_loss).backward()
            scaler_g.step(opt_g); scaler_g.update()
            run_g += g_loss.item() * deg.size(0)
            run_d += d_loss_v * deg.size(0)
            n     += deg.size(0)
            pbar.set_postfix(G=f"{run_g/n:.4f}",
                             D=f"{run_d/n:.4f}",
                             d_state='train' if train_d_this_epoch else 'frozen')
        print(f"  ep {ep+1}: G={run_g/n:.4f}  D={run_d/n:.4f}  d_state={'train' if train_d_this_epoch else 'frozen'}")
    torch.save({'state_dict': G.state_dict(),
                'adv_w': adv_w,
                'freeze_d_after': freeze_d_after}, SWINIR_CKPT)
    print(f"Saved -> {SWINIR_CKPT}")


train_swinir_gan(epochs=8, batch_size=4, lr_g=1e-4, lr_d=1e-4,
                 adv_w=0.005, freeze_d_after=4)


# ---- Inference ----
_swin_net = None
def _load_swinir():
    global _swin_net
    if _swin_net is None:
        _swin_net = _make_swinir().to(DEVICE).eval()
        sd = torch.load(SWINIR_CKPT, map_location=DEVICE, weights_only=False)
        _swin_net.load_state_dict(sd['state_dict'])
    return _swin_net


@torch.no_grad()
def enhance_swinir_gan(img_pil):
    """One-shot SwinIR-GAN restoration (no iteration, no kind conditioning)."""
    net = _load_swinir()
    orig = img_pil.size
    src = img_pil.convert('RGB').resize((SWINIR_SIZE, SWINIR_SIZE), Image.BILINEAR)
    x = T.ToTensor()(src).unsqueeze(0).to(DEVICE)
    with autocast(enabled=True):
        y = net(x).clamp(0, 1)
    arr = (y.squeeze().permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype('uint8')
    return Image.fromarray(arr).resize(orig)


print("SwinIR + GAN v6 restorer ready.")


## V2 — Step 3 — CycleGAN-CBAM (Phase 4 GAN #2)

Professor deliverable #2a. The original Phase 4 had only one GAN-based restorer (A-ESRGAN).
This adds CycleGAN with CBAM attention as the second GAN-based approach. CycleGAN is well
suited because it works with the unpaired degraded↔clean translation framing — we can use
arbitrary degraded fundus images as domain A and pristine APTOS as domain B, even when
specific (clean, degraded) pairs are not perfectly aligned.


In [ ]:
# [RESUME] CycleGAN disabled by request (not training/using it).
print('[resume] CycleGAN disabled -- removed from enhancers.')


## V2 — Step 4 — Conditional DDPM (vanilla diffusion, Phase 4 #2)

Professor deliverable #2b. The existing diffusion restorer (Cold Diffusion) uses a
non-standard forward process — the Phase 1 degradation operators as the "noising"
operator. The professor specifically asked for a vanilla DDPM-style model (Gaussian
noise forward process). This implements a conditional DDPM where the denoising
U-Net is conditioned on the degraded image (concatenated as extra channels) so the
network learns to recover the clean image from pure noise *given* the degraded view.


In [ ]:
# === V2 PATCH (Step 4): conditional vanilla DDPM ===
import math
import torch.nn as nn
import torch.nn.functional as F

DDPM_CKPT = CHECKPOINTS_DIR / 'ddpm_fundus_v1.pt'
DDPM_SIZE = 256


class _SinusoidalPE(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000.0) / max(half - 1, 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class _DDPMUNet(nn.Module):
    """U-Net conditioned on (timestep, degraded image).
    Input  : concat(noisy_clean, degraded) -> 6 channels.
    Output : predicted noise -> 3 channels.
    """
    def __init__(self, in_ch=6, out_ch=3, base_ch=64, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            _SinusoidalPE(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim),
        )
        def _block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.GroupNorm(8, out_c),
                nn.GELU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.GroupNorm(8, out_c),
                nn.GELU(),
            )
        self.enc1 = _block(in_ch, base_ch)
        self.enc2 = _block(base_ch, base_ch * 2)
        self.enc3 = _block(base_ch * 2, base_ch * 4)
        self.mid  = _block(base_ch * 4, base_ch * 4)
        self.film = nn.Linear(time_dim, base_ch * 4 * 2)
        self.dec3 = _block(base_ch * 8, base_ch * 2)
        self.dec2 = _block(base_ch * 4, base_ch)
        self.dec1 = _block(base_ch * 2, base_ch)
        self.final = nn.Conv2d(base_ch, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m  = self.mid(self.pool(e3))
        scale, shift = self.film(t_emb).unsqueeze(-1).unsqueeze(-1).chunk(2, dim=1)
        m = m * (1 + scale) + shift
        d3 = self.dec3(torch.cat([self.up(m),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return self.final(d1)


class FundusDDPM:
    """Standard DDPM (Ho et al. 2020) conditioned on the degraded image."""
    def __init__(self, model, T=1000, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.model = model
        self.T = T
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1 - self.alpha_cumprod)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_acp   = self.sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_omacp = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_acp * x0 + sqrt_omacp * noise, noise

    def train_step(self, clean, degraded, optimizer, scaler=None):
        B = clean.size(0)
        t = torch.randint(0, self.T, (B,), device=self.device)
        noise = torch.randn_like(clean)
        noisy_clean, _ = self.q_sample(clean, t, noise)
        inp = torch.cat([noisy_clean, degraded], dim=1)   # (B, 6, H, W)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast(enabled=True):
                pred = self.model(inp, t)
                loss = F.mse_loss(pred, noise)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        else:
            pred = self.model(inp, t)
            loss = F.mse_loss(pred, noise)
            loss.backward(); optimizer.step()
        return loss.item()

    @torch.no_grad()
    def restore(self, degraded, n_steps=50):
        B = degraded.size(0)
        x = torch.randn(B, 3, degraded.size(2), degraded.size(3), device=self.device)
        step_size = max(self.T // n_steps, 1)
        timesteps = list(range(self.T - 1, 0, -step_size))
        for t_val in timesteps:
            t = torch.full((B,), t_val, device=self.device, dtype=torch.long)
            inp = torch.cat([x, degraded], dim=1)
            pred_noise = self.model(inp, t)
            alpha = self.alphas[t_val]
            alpha_cumprod = self.alpha_cumprod[t_val]
            beta = self.betas[t_val]
            x = (1 / torch.sqrt(alpha)) * (
                x - (beta / torch.sqrt(1 - alpha_cumprod)) * pred_noise
            )
            if t_val > 1:
                x = x + torch.sqrt(beta) * torch.randn_like(x)
        return x.clamp(-1, 1)


class _DDPMDataset(Dataset):
    def __init__(self, df, n_per=4):
        self.df = df.reset_index(drop=True); self.n_per = n_per
        self.tfm = T.Compose([T.Resize((DDPM_SIZE, DDPM_SIZE)),
                              T.ToTensor(),
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    def __len__(self): return len(self.df) * self.n_per
    def __getitem__(self, i):
        row = self.df.iloc[i // self.n_per]
        try:
            img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (DDPM_SIZE, DDPM_SIZE))
        img = img.resize((DDPM_SIZE, DDPM_SIZE), Image.BILINEAR)
        kind  = random.choice(['blur', 'exposure', 'noise'])
        level = random.choice(['low', 'mid', 'high'])
        deg   = DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])
        return self.tfm(deg), self.tfm(img)


def train_ddpm(epochs=15, batch_size=8, lr=1e-4, T=1000):
    """Resume-guarded conditional DDPM training (~3-6 hr on A100)."""
    if DDPM_CKPT.exists():
        print(f"[skip] {DDPM_CKPT.name} already on Drive — not retraining.")
        return
    print(f"Training conditional DDPM ({epochs} epochs, T={T}, batch={batch_size}, lr={lr})...")
    df  = pd.read_csv(PRISTINE_CSV)
    ds  = _DDPMDataset(df, n_per=4)
    dl  = DataLoader(ds, batch_size=batch_size, shuffle=True,
                     num_workers=2, pin_memory=True)
    net = _DDPMUNet().to(DEVICE)
    ddpm = FundusDDPM(net, T=T, device=DEVICE)
    opt  = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-5)
    scaler = GradScaler(enabled=True)
    for ep in range(epochs):
        net.train()
        running, n = 0.0, 0
        pbar = tqdm(dl, desc=f"ddpm ep {ep+1}/{epochs}", leave=False)
        for deg, clean in pbar:
            deg   = deg.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            loss = ddpm.train_step(clean, deg, opt, scaler=scaler)
            running += loss * deg.size(0); n += deg.size(0)
            pbar.set_postfix(MSE=f"{running/n:.4f}")
        print(f"  ep {ep+1}: MSE={running/n:.4f}")
    torch.save({'state_dict': net.state_dict(), 'T': T}, DDPM_CKPT)
    print(f"Saved -> {DDPM_CKPT}")


train_ddpm(epochs=15, batch_size=8, lr=1e-4, T=1000)


# ---- Inference ----
_ddpm_net = None
_ddpm     = None
def _load_ddpm(T=1000):
    global _ddpm_net, _ddpm
    if _ddpm is not None:
        return _ddpm
    _ddpm_net = _DDPMUNet().to(DEVICE).eval()
    sd = torch.load(DDPM_CKPT, map_location=DEVICE, weights_only=False)
    _ddpm_net.load_state_dict(sd['state_dict'])
    _ddpm = FundusDDPM(_ddpm_net, T=sd.get('T', T), device=DEVICE)
    return _ddpm


@torch.no_grad()
def enhance_ddpm(img_pil, n_steps=50):
    """Restore via conditional DDPM with DDIM-style fast sampling."""
    ddpm = _load_ddpm()
    orig = img_pil.size
    src  = img_pil.convert('RGB').resize((DDPM_SIZE, DDPM_SIZE), Image.BILINEAR)
    tfm  = T.Compose([T.ToTensor(),
                      T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    x = tfm(src).unsqueeze(0).to(DEVICE)
    y = ddpm.restore(x, n_steps=n_steps)
    arr = (y.squeeze().permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
    arr = (arr * 255).clip(0, 255).astype('uint8')
    return Image.fromarray(arr).resize(orig)


print("Conditional DDPM restorer ready.")


## V3 — Step 7 — Pathology-preserving conditional DDPM (Phase 4 Diffusion #3)

Upgrades the vanilla DDPM (Step 4) with three explicit pathology-preservation
mechanisms, so the restorer recovers image quality **without inventing or
erasing lesions** (microaneurysms, haemorrhages, exudates):

1. **Pathology-aware perceptual loss** — computed in the feature space of *our*
   trained EfficientNetV2-S DR classifier (`build_model_v3('efficientnet_b3')`),
   not VGG/ImageNet. The model is penalised for changing the features the DR
   classifier treats as diagnostic.
2. **Input-fidelity loss** — an L1 term anchoring the reconstructed clean image
   to the degraded observation.
3. **Sampling-time anti-hallucination clamp** — at every reverse step the
   predicted `x0` is blended toward the degraded input (`clamp_strength`),
   capping how far the sampler may drift from the evidence.

The forward/denoising objective is still standard DDPM (Gaussian noise,
epsilon-prediction) — identical to Step 4 — so vanilla-vs-pathology is a fair
ablation. Kept as a **separate** enhancer `'ddpm_path'` ("DDPM (Pathology)")
with its own checkpoint `ddpm_pathology_v1.pt`; the recovery table now reports
both DDPMs side by side. Training is resume-guarded (runs once).


In [ ]:
# === V3 PATCH (Step 7): pathology-preserving conditional DDPM ===
# Phase 4 Diffusion #3. Vanilla DDPM objective (Step 4) + three pathology
# preservation mechanisms: classifier-feature perceptual loss, input-fidelity
# L1, and a sampling-time anti-hallucination clamp. Kept alongside 'ddpm'.
import math, random
import torch, torch.nn as nn, torch.nn.functional as F

DDPM_PATH_CKPT = CHECKPOINTS_DIR / 'ddpm_pathology_v1.pt'
DDPM_PATH_SIZE = 256
_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class _PathSinPE(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000.0) / max(half - 1, 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class _PathAttn(nn.Module):
    """Self-attention at the bottleneck for global context."""
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        self.scale = ch ** -0.5
    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(B, 3, C, H * W).unbind(1)
        attn = torch.softmax((q.transpose(-1, -2) @ k) * self.scale, dim=-1)
        out  = (v @ attn.transpose(-1, -2)).reshape(B, C, H, W)
        return x + self.proj(out)


class _PathUNet(nn.Module):
    """Conditional U-Net: concat(noisy_clean, degraded)=6ch -> predicted noise 3ch.
    FiLM modulation on the timestep + self-attention at the bottleneck."""
    def __init__(self, in_ch=6, out_ch=3, base_ch=64, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            _PathSinPE(time_dim),
            nn.Linear(time_dim, time_dim), nn.GELU(),
            nn.Linear(time_dim, time_dim))
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.GroupNorm(8, o), nn.GELU(),
                nn.Conv2d(o, o, 3, padding=1), nn.GroupNorm(8, o), nn.GELU())
        self.enc1 = blk(in_ch, base_ch)
        self.enc2 = blk(base_ch, base_ch * 2)
        self.enc3 = blk(base_ch * 2, base_ch * 4)
        self.mid  = blk(base_ch * 4, base_ch * 4)
        self.mid_attn = _PathAttn(base_ch * 4)
        self.film = nn.Linear(time_dim, base_ch * 4 * 2)
        self.dec3 = blk(base_ch * 8, base_ch * 2)
        self.dec2 = blk(base_ch * 4, base_ch)
        self.dec1 = blk(base_ch * 2, base_ch)
        self.final = nn.Conv2d(base_ch, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    def forward(self, x, t):
        te = self.time_mlp(t)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m  = self.mid(self.pool(e3))
        scale, shift = self.film(te).unsqueeze(-1).unsqueeze(-1).chunk(2, dim=1)
        m  = self.mid_attn(m * (1 + scale) + shift)
        d3 = self.dec3(torch.cat([self.up(m),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return self.final(d1)


class PathologyPerceptualLoss(nn.Module):
    """Perceptual loss in the feature space of OUR trained DR classifier
    (MultiScaleClassifier wrapping a features_only EfficientNetV2-S).
    Inputs are in DDPM space ([-1,1]); converted to the classifier's
    ImageNet-normalised 384px space internally."""
    def __init__(self, device=DEVICE):
        super().__init__()
        clf = build_model_v3('efficientnet_b3', pretrained=False).to(device)
        ck  = CKPT_DIR_V3 / 'efficientnet_b3_v3_best.pt'
        sd  = torch.load(ck, map_location=device, weights_only=False)
        clf.load_state_dict(sd['state_dict'] if 'state_dict' in sd else sd, strict=False)
        clf.eval()
        self.backbone = clf.backbone               # features_only -> list of maps
        self.n_stages = min(3, clf.n_stages)
        for p in self.parameters():
            p.requires_grad_(False)
        self.register_buffer('mean', _IMAGENET_MEAN.to(device))
        self.register_buffer('std',  _IMAGENET_STD.to(device))
    def _feats(self, x):
        x01 = (x * 0.5 + 0.5).clamp(0, 1)
        x01 = F.interpolate(x01, size=IMAGE_SIZE_V3, mode='bilinear', align_corners=False)
        x01 = (x01 - self.mean) / self.std
        return list(self.backbone(x01))[-self.n_stages:]
    def forward(self, restored, clean):
        fr, fc = self._feats(restored), self._feats(clean)
        return sum(F.l1_loss(a, b) for a, b in zip(fr, fc)) / len(fr)


class PathologyDDPM:
    """Standard DDPM (Ho et al. 2020) conditioned on the degraded image, with a
    pathology-preservation clamp during sampling."""
    def __init__(self, model, T=1000, beta_start=1e-4, beta_end=0.02, device=DEVICE):
        self.model = model; self.T = T; self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.acp = torch.cumprod(self.alphas, dim=0)
        self.sqrt_acp = torch.sqrt(self.acp)
        self.sqrt_omacp = torch.sqrt(1 - self.acp)
    def q_sample(self, x0, t, noise):
        a = self.sqrt_acp[t].view(-1, 1, 1, 1)
        b = self.sqrt_omacp[t].view(-1, 1, 1, 1)
        return a * x0 + b * noise
    def _x0_from_eps(self, xt, t, eps):
        a = self.sqrt_acp[t].view(-1, 1, 1, 1)
        b = self.sqrt_omacp[t].view(-1, 1, 1, 1)
        return (xt - b * eps) / a
    @torch.no_grad()
    def restore(self, degraded, n_steps=50, clamp_strength=0.15):
        self.model.eval()
        B = degraded.size(0)
        x = torch.randn(B, 3, degraded.size(2), degraded.size(3), device=self.device)
        step = max(self.T // n_steps, 1)
        ts = list(range(self.T - 1, 0, -step))
        for i, tv in enumerate(ts):
            t = torch.full((B,), tv, device=self.device, dtype=torch.long)
            eps = self.model(torch.cat([x, degraded], dim=1), t)
            x0 = self._x0_from_eps(x, t, eps)
            # ---- pathology-preservation clamp: anchor x0 toward evidence ----
            x0 = ((1 - clamp_strength) * x0 + clamp_strength * degraded).clamp(-1, 1)
            tv_next = ts[i + 1] if i + 1 < len(ts) else 0
            if tv_next > 0:
                a_next = self.acp[tv_next]
                x = torch.sqrt(a_next) * x0 + torch.sqrt(1 - a_next) * torch.randn_like(x)
            else:
                x = x0
        return x.clamp(-1, 1)


class _PathDataset(Dataset):
    """(degraded, clean) pairs; degradation generated on the fly with the
    Phase-1 operators, matching the vanilla DDPM dataset. Data in [-1,1]."""
    def __init__(self, df, n_per=4):
        self.df = df.reset_index(drop=True); self.n_per = n_per
        self.tfm = T.Compose([T.Resize((DDPM_PATH_SIZE, DDPM_PATH_SIZE)),
                              T.ToTensor(),
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    def __len__(self): return len(self.df) * self.n_per
    def __getitem__(self, i):
        row = self.df.iloc[i // self.n_per]
        try:
            img = Image.open(resolve_image(row['id_code'])).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (DDPM_PATH_SIZE, DDPM_PATH_SIZE))
        img = img.resize((DDPM_PATH_SIZE, DDPM_PATH_SIZE), Image.BILINEAR)
        kind  = random.choice(['blur', 'exposure', 'noise'])
        level = random.choice(['low', 'mid', 'high'])
        deg   = DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])
        return self.tfm(deg), self.tfm(img)


def train_ddpm_pathology(epochs=15, batch_size=8, lr=1e-4, T=1000,
                         lambda_perc=0.1, lambda_fid=0.05):
    """Resume-guarded. Loss = MSE(eps) + lambda_perc * PathologyPerceptual(x0, clean)
                                       + lambda_fid  * L1(x0, degraded)."""
    if DDPM_PATH_CKPT.exists():
        print(f"[skip] {DDPM_PATH_CKPT.name} already on Drive — not retraining.")
        return
    print(f"Training pathology DDPM ({epochs} ep, T={T}, bs={batch_size}, lr={lr}, "
          f"lambda_perc={lambda_perc}, lambda_fid={lambda_fid})...")
    df = pd.read_csv(PRISTINE_CSV)
    dl = DataLoader(_PathDataset(df, n_per=4), batch_size=batch_size, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    net  = _PathUNet().to(DEVICE)
    ddpm = PathologyDDPM(net, T=T, device=DEVICE)
    try:
        perc = PathologyPerceptualLoss(DEVICE)
        print("  pathology perceptual loss: ENABLED (EfficientNetV2-S DR features)")
    except Exception as e:
        perc = None
        print(f"  WARNING: perceptual loss disabled ({e}); using MSE + fidelity only")
    opt   = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = GradScaler(enabled=True)
    for ep in range(epochs):
        net.train()
        run = dict(tot=0.0, mse=0.0, perc=0.0, fid=0.0, n=0)
        pbar = tqdm(dl, desc=f"path-ddpm ep {ep+1}/{epochs}", leave=False)
        for deg, clean in pbar:
            deg   = deg.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            B = clean.size(0)
            t = torch.randint(0, T, (B,), device=DEVICE)
            noise = torch.randn_like(clean)
            noisy = ddpm.q_sample(clean, t, noise)
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=True):
                eps = net(torch.cat([noisy, deg], dim=1), t)
                l_mse = F.mse_loss(eps, noise)
                x0 = ddpm._x0_from_eps(noisy, t, eps).clamp(-1, 1)
                l_perc = perc(x0, clean) if (perc is not None and lambda_perc > 0) \
                         else torch.zeros((), device=DEVICE)
                l_fid  = F.l1_loss(x0, deg) if lambda_fid > 0 \
                         else torch.zeros((), device=DEVICE)
                loss = l_mse + lambda_perc * l_perc + lambda_fid * l_fid
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            run['tot']  += loss.item() * B
            run['mse']  += l_mse.item() * B
            run['perc'] += float(l_perc) * B
            run['fid']  += float(l_fid) * B
            run['n']    += B
            pbar.set_postfix(tot=f"{run['tot']/run['n']:.3f}",
                             mse=f"{run['mse']/run['n']:.3f}")
        sched.step()
        n = run['n']
        print(f"  ep {ep+1}: tot={run['tot']/n:.4f} mse={run['mse']/n:.4f} "
              f"perc={run['perc']/n:.4f} fid={run['fid']/n:.4f}")
    torch.save({'state_dict': net.state_dict(), 'T': T}, DDPM_PATH_CKPT)
    print(f"Saved -> {DDPM_PATH_CKPT}")


train_ddpm_pathology(epochs=15, batch_size=8, lr=1e-4, T=1000)


# ---- Inference ----
_path_net  = None
_path_ddpm = None
def _load_path_ddpm(T=1000):
    global _path_net, _path_ddpm
    if _path_ddpm is not None:
        return _path_ddpm
    _path_net = _PathUNet().to(DEVICE).eval()
    sd = torch.load(DDPM_PATH_CKPT, map_location=DEVICE, weights_only=False)
    _path_net.load_state_dict(sd['state_dict'])
    _path_ddpm = PathologyDDPM(_path_net, T=sd.get('T', T), device=DEVICE)
    return _path_ddpm


@torch.no_grad()
def enhance_ddpm_path(img_pil, n_steps=50, clamp_strength=0.15):
    """Restore via the pathology-preserving DDPM. Same PIL->PIL signature as
    enhance_ddpm so the build_enhanced registry call is a drop-in."""
    ddpm = _load_path_ddpm()
    orig = img_pil.size
    src  = img_pil.convert('RGB').resize((DDPM_PATH_SIZE, DDPM_PATH_SIZE), Image.BILINEAR)
    tfm  = T.Compose([T.ToTensor(), T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    x = tfm(src).unsqueeze(0).to(DEVICE)
    y = ddpm.restore(x, n_steps=n_steps, clamp_strength=clamp_strength)
    arr = (y.squeeze().permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
    arr = (arr * 255).clip(0, 255).astype('uint8')
    return Image.fromarray(arr).resize(orig)


print("Pathology-preserving DDPM restorer ready.")


In [ ]:
# 4.3 Build enhanced versions of every degraded image (test ids only)
test_id_set = set(pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].astype(str))

def build_enhanced(method_name, fn):
    """fn signature: fn(img_pil, kind, level) -> PIL.Image"""
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            src_dir = DEGRADED_DIR / k / l
            out_dir = ENHANCED_DIR / method_name / k / l
            out_dir.mkdir(parents=True, exist_ok=True)
            mani = pd.read_csv(src_dir / 'manifest.csv')
            mani = mani[mani['id_code'].astype(str).isin(test_id_set)]
            for _, row in tqdm(mani.iterrows(), total=len(mani),
                                desc=f'{method_name}/{k}/{l}', leave=False):
                out = out_dir / row['rel_path']
                if not out.exists():
                    src = Image.open(src_dir / row['rel_path']).convert('RGB')
                    fn(src, k, l).save(out)
            mani.assign(method=method_name).to_csv(out_dir / 'manifest.csv', index=False)

build_enhanced('clahe',     lambda im, k, l: enhance_clahe(im))
build_enhanced('genai',     lambda im, k, l: enhance_genai(im))
build_enhanced('cold_diff',  lambda im, k, l: enhance_cold_diffusion(im, kind=k,
                                                                      t_start=COLD_T_FOR_LEVEL[l]))
build_enhanced('swinir_gan', lambda im, k, l: enhance_swinir_gan(im))
build_enhanced('ddpm',       lambda im, k, l: enhance_ddpm(im))       # V2 NEW (Step 4)
build_enhanced('ddpm_path',  lambda im, k, l: enhance_ddpm_path(im))  # V3 NEW (Step 7)

# Cache enhanced images to Drive (mirrors cell 24's cache_degraded tar pattern).
# Eliminates ~15 min rebuild on session reload (PROJECT_OVERVIEW §11.5).
_ENH_CACHE = Path('/content/drive/MyDrive/Thesis/cache_enhanced_v3.tar.gz')
if not _ENH_CACHE.exists():
    print(f'Caching enhanced tree to Drive -> {_ENH_CACHE} ...')
    import subprocess
    subprocess.run(['tar', '-czf', str(_ENH_CACHE),
                    '-C', str(ENHANCED_DIR.parent),
                    ENHANCED_DIR.name], check=False)
    print('  done.')
else:
    print(f'[skip] enhanced cache already exists on Drive -> {_ENH_CACHE}')


In [ ]:
# 4.4 Re-evaluate the three classifiers on raw degraded vs every enhancer
ENHANCERS = ('clahe', 'genai', 'cold_diff', 'swinir_gan', 'ddpm', 'ddpm_path')
VARIANTS  = ('raw', *ENHANCERS)

# Display-friendly labels for plots and tables (cf. master prompt §6 notes).
ENHANCER_LABELS = {
    'raw':         'raw (no restoration)',
    'clahe':       'CLAHE',
    'genai':       'A-ESRGAN',
    'cold_diff':   'Cold Diffusion',
    'swinir_gan':  'SwinIR-GAN',
    'ddpm':        'DDPM (Vanilla)',
    'ddpm_path':   'DDPM (Pathology)',
}

rows = []
for name in MODEL_NAMES:
    model = models[name]
    print(f'\n=== {name} ===')
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            for variant in VARIANTS:
                root = DEGRADED_DIR / k / l if variant == 'raw' else ENHANCED_DIR / variant / k / l
                if not (root / 'manifest.csv').exists():
                    continue
                ds = FolderDataset(root, transform=TFM_EVAL)
                ds.df = ds.df[ds.df['id_code'].astype(str).isin(test_id_set)].reset_index(drop=True)
                m = evaluate(model, DataLoader(ds, batch_size=32, num_workers=2, pin_memory=True))
                rows.append({'model': name, 'degradation': k, 'level': l, 'variant': variant, **m})
                print(f'  {k}/{l}/{variant}: acc={m["accuracy"]:.4f}')
    del model; torch.cuda.empty_cache()

rec_df = pd.DataFrame(rows)
rec_df.to_csv(P4 / 'metrics' / 'recovery_accuracy.csv', index=False)
rec_df.head()


In [ ]:
# 4.5 Recovery plots — accuracy
level_order_no_none = ['low', 'mid', 'high']
for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.2*len(MODEL_NAMES), 4), sharey=True)
    for ax, name in zip(axes, MODEL_NAMES):
        sub = rec_df[(rec_df['degradation'] == kind) & (rec_df['model'] == name)].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order_no_none, ordered=True)
        for variant in VARIANTS:
            d = sub[sub['variant'] == variant].sort_values('level')
            ax.plot(d['level'], d['accuracy'], marker='o', label=variant)
        ax.set_title(name); ax.set_xlabel('level'); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
    axes[0].set_ylabel('accuracy'); axes[-1].legend()
    fig.suptitle(f'Accuracy recovery — {kind}')
    plt.tight_layout()
    out = P4 / 'plots' / f'recovery_accuracy_{kind}.png'
    plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)


In [ ]:
# 4.6 Slim XAI re-evaluation on enhanced images
METHOD_FOR = {'resnet50': ('gradcam', gradcam_heatmap),
               'efficientnet_b3': ('gradcam', gradcam_heatmap),
               'vit_base': ('attention', attention_rollout)}
EXPLAIN_IDS_P4 = list(test_id_set)[:15]

rows = []
for name in MODEL_NAMES:
    method_name, fn = METHOD_FOR[name]
    model = load_classifier(name)
    print(f'\n=== {name} / {method_name} ===')
    for id_code in tqdm(EXPLAIN_IDS_P4):
        if id_code not in labels_df.index: continue
        label = int(labels_df[id_code])
        try:
            x_c = load_tensor(resolve_image(id_code))    # <-- was hardcoded .png
        except FileNotFoundError:
            continue
        try: h_c = fn(model, x_c, label)
        except Exception: continue
        for k in DEGRADATION_TYPES:
            for l in DEGRADATION_LEVELS:
                for variant in ('raw', *ENHANCERS):
                    base = DEGRADED_DIR if variant == 'raw' else ENHANCED_DIR / variant
                    try:
                        src = find_in_folder(base / k / l, id_code)   # <-- ext-agnostic
                    except FileNotFoundError:
                        continue
                    x = load_tensor(src)
                    try: h = fn(model, x, label)
                    except Exception: continue
                    rows.append({'id_code': id_code, 'model': name, 'method': method_name,
                                 'degradation': k, 'level': l, 'variant': variant,
                                 'stability': stability_spearman(h_c, h),
                                 'insertion_auc': insertion_auc(model, x, h, label)})
    del model; torch.cuda.empty_cache()

xai_rec = pd.DataFrame(rows)
xai_rec.to_csv(P4 / 'metrics' / 'recovery_xai.csv', index=False)
xai_rec.head()

In [ ]:
# 4.7 Recovery plots — XAI  (dynamic over ENHANCERS; qualitative grid widens)
for metric in ('stability', 'insertion_auc'):
    for kind in DEGRADATION_TYPES:
        sub = xai_rec[xai_rec['degradation'] == kind].copy()
        if sub.empty: continue
        sub['level'] = pd.Categorical(sub['level'], categories=level_order_no_none, ordered=True)
        g = sub.groupby(['model', 'variant', 'level'])[metric].mean().reset_index()
        fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.2*len(MODEL_NAMES), 4), sharey=True)
        for ax, name in zip(axes, MODEL_NAMES):
            for variant in VARIANTS:
                d = g[(g['model'] == name) & (g['variant'] == variant)].sort_values('level')
                ax.plot(d['level'], d[metric], marker='o', label=variant)
            ax.set_title(name); ax.grid(alpha=0.3); ax.set_xlabel('level')
        axes[0].set_ylabel(metric); axes[-1].legend()
        fig.suptitle(f'XAI {metric} recovery — {kind}'); plt.tight_layout()
        out = P4 / 'plots' / f'recovery_xai_{metric}_{kind}.png'
        plt.savefig(out, dpi=150); plt.show()

# Side-by-side qualitative — one row per degradation, one column per variant
demo_id = EXPLAIN_IDS_P4[0]
n_cols  = 2 + len(ENHANCERS)   # clean, degraded, + each enhancer
fig, axes = plt.subplots(len(DEGRADATION_TYPES), n_cols,
                         figsize=(2.8*n_cols, 3.2*len(DEGRADATION_TYPES)))
for r, kind in enumerate(DEGRADATION_TYPES):
    clean = Image.open(resolve_image(demo_id)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    deg   = Image.open(find_in_folder(DEGRADED_DIR / kind / 'high', demo_id)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    panels = [(clean, 'clean'), (deg, f'{kind}-high')]
    for variant in ENHANCERS:
        try:
            img = Image.open(find_in_folder(ENHANCED_DIR / variant / kind / 'high', demo_id)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
            panels.append((img, variant))
        except FileNotFoundError:
            panels.append((Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE)), f'{variant} (missing)'))
    for ax, (im, ttl) in zip(axes[r], panels):
        ax.imshow(im); ax.set_title(ttl); ax.axis('off')
plt.tight_layout()
out = P4 / 'samples' / f'recovery_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)


In [ ]:
P5 = PHASE_DIRS['phase5_quality_ensemble']

# 5.1 Train (or load) an EyeQ quality classifier (good/usable/reject)
class EyeQDataset(Dataset):
    def __init__(self, csv_path, images_root, transform,
                 id_col='image', quality_col='quality'):
        self.df = pd.read_csv(csv_path); self.df[id_col] = self.df[id_col].astype(str)
        self.images_root = Path(images_root); self.transform = transform
        self.id_col, self.quality_col = id_col, quality_col
        all_imgs = {p.stem: p for p in self.images_root.rglob('*.*')
                    if p.suffix.lower() in ('.png', '.jpg', '.jpeg')}
        self.df['__path'] = self.df[id_col].apply(lambda s: all_imgs.get(Path(s).stem))
        self.df = self.df.dropna(subset=['__path']).reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['__path']).convert('RGB')
        return self.transform(img), int(row[self.quality_col])

In [ ]:
def build_quality_model(): return timm.create_model('resnet18', pretrained=True, num_classes=3)
Q_CKPT = CHECKPOINTS_DIR / 'quality_classifier.pt'

In [ ]:
def train_quality(epochs=2, bs=32):
    if EYEQ_CSV is None: return None
    ds = EyeQDataset(EYEQ_CSV, EYEQ_DIR, TFM_TRAIN)
    n_val = max(1, int(0.15 * len(ds)))
    tds, vds = random_split(ds, [len(ds)-n_val, n_val], generator=torch.Generator().manual_seed(0))
    vds.dataset = EyeQDataset(EYEQ_CSV, EYEQ_DIR, TFM_EVAL)
    tl = DataLoader(tds, batch_size=bs, shuffle=True,  num_workers=2)
    vl = DataLoader(vds, batch_size=bs, shuffle=False, num_workers=2)
    mdl = build_quality_model().to(DEVICE)
    optim = torch.optim.AdamW(mdl.parameters(), lr=3e-4, weight_decay=1e-4); crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        mdl.train()
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optim.zero_grad(); crit(mdl(x), y).backward(); optim.step()
        mdl.eval(); ys, ps = [], []
        with torch.no_grad():
            for x, y in vl:
                ps.append(mdl(x.to(DEVICE)).argmax(1).cpu().numpy()); ys.append(y.numpy())
        y = np.concatenate(ys); p = np.concatenate(ps)
        print(f'epoch {ep+1}: val acc = {accuracy_score(y, p):.4f}')
    rep = classification_report(y, p, target_names=list(EYEQ_QUALITY_LABELS.values()),
                                 output_dict=True, zero_division=0)
    torch.save(mdl.state_dict(), Q_CKPT)
    with open(P5 / 'metrics' / 'quality_classifier_metrics.json', 'w') as f:
        json.dump(rep, f, indent=2)
    return mdl

In [ ]:
if Q_CKPT.exists():
    qmodel = build_quality_model().to(DEVICE)
    qmodel.load_state_dict(torch.load(Q_CKPT, map_location=DEVICE)); qmodel.eval()
    print('Loaded existing quality classifier.')
else:
    qmodel = train_quality()

In [ ]:
# 5.2 Pick best_clean and best_robust from Phase 2 results
stress_df = pd.read_csv(P2 / 'metrics' / 'stress_test_results.csv')
best_clean  = stress_df[stress_df['degradation'] == 'clean'].sort_values('accuracy', ascending=False).iloc[0]['model']
best_robust = stress_df[stress_df['degradation'] != 'clean'].groupby('model')['accuracy'].mean().idxmax()
print('best_clean :', best_clean)
print('best_robust:', best_robust)

ROUTES = {
    'good':   {'enhancement': 'none',  'model': best_clean,  'xai': 'attention' if best_clean  == 'vit_base' else 'gradcam'},
    'usable': {'enhancement': 'clahe', 'model': best_clean,  'xai': 'attention' if best_clean  == 'vit_base' else 'gradcam'},
    'reject': {'enhancement': 'genai', 'model': best_robust, 'xai': 'attention' if best_robust == 'vit_base' else 'gradcam'},
}
ROUTES

In [ ]:
classifiers = {n: load_classifier(n) for n in {best_clean, best_robust}}
USE_PRECOMPUTED_GENAI = (ENHANCED_DIR / 'genai').exists()

def predict_quality(img):
    if qmodel is None: return 'good'
    x = TFM_EVAL(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return EYEQ_QUALITY_LABELS[int(qmodel(x).argmax(1).item())]

def apply_enhancement(img, kind, kind_hint=None, level_hint=None, id_hint=None):
    if kind == 'none':  return img
    if kind == 'clahe': return enhance_clahe(img)
    if kind == 'genai':
        if USE_PRECOMPUTED_GENAI and kind_hint and level_hint and id_hint:
            cached = ENHANCED_DIR / 'genai' / kind_hint / level_hint / f'{id_hint}.png'
            if cached.exists(): return Image.open(cached).convert('RGB')
        return enhance_genai(img)
    return img

XAI_FNS = {'gradcam': gradcam_heatmap, 'attention': attention_rollout_v3}

def trust_score(conf, insertion): return float(0.5 * conf + 0.5 * insertion)

def pipeline(img, kind_hint=None, level_hint=None, id_hint=None):
    quality  = predict_quality(img)
    route    = ROUTES[quality]
    enhanced = apply_enhancement(img, route['enhancement'],
                                  kind_hint=kind_hint, level_hint=level_hint, id_hint=id_hint)
    model = classifiers[route['model']]
    x = TFM_EVAL(enhanced).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(x), 1)[0]
        pred = int(prob.argmax().item()); conf = float(prob[pred].item())
    heat = XAI_FNS[route['xai']](model, x, target_class=pred)
    ins  = insertion_auc(model, x, heat, pred)
    return {'quality_pred': quality, 'route': route, 'pred': pred, 'conf': conf,
            'insertion_auc': ins, 'trust': trust_score(conf, ins),
            'heatmap': heat, 'enhanced_img': enhanced}

## V2 — Step 1d — Quality classifier threshold calibration

The Phase 5 quality classifier (`Q_CKPT`) over-rejects on clean APTOS — Counter from cell 87
showed almost every clean test image labelled `reject`. Argmax routing on a class-imbalanced
output is the cause. Below we sweep thresholds on the softmax outputs over clean APTOS and pick
the operating point where the routing distribution is closest to the target ~60:25:15 mix of
good/usable/reject. Then we replace `predict_quality()` to use thresholds instead of argmax.


In [ ]:
# === V2 PATCH (Step 1d): threshold calibration for the quality classifier ===
# Sweep softmax probability thresholds on clean APTOS until we get ~60:25:15.

from collections import Counter
import numpy as np

P5 = PHASE_DIRS['phase5_quality_ensemble']
P5_METRICS = P5 / 'metrics'
P5_METRICS.mkdir(parents=True, exist_ok=True)

# 1) Score clean APTOS test images
test_ids_q = pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].astype(str).tolist()
probs_rows = []
for sid in tqdm(test_ids_q, desc='quality probs'):
    try:
        img = Image.open(resolve_image(sid)).convert('RGB')
    except FileNotFoundError:
        continue
    x = TFM_EVAL(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(qmodel(x), 1)[0].cpu().numpy()
    probs_rows.append({'id_code': sid, 'p_good': p[0], 'p_usable': p[1], 'p_reject': p[2]})

probs_df = pd.DataFrame(probs_rows)
probs_df.to_csv(P5_METRICS / 'quality_clean_probs.csv', index=False)
print(f'Scored {len(probs_df)} clean APTOS test images.')

# 2) Search thresholds: minimise |observed - target| across the routing distribution.
TARGET = {'good': 0.60, 'usable': 0.25, 'reject': 0.15}

def _route_under(p_good_thr, p_usable_thr, df):
    """If p_good > p_good_thr -> good. Else if p_usable > p_usable_thr -> usable. Else reject."""
    labels = []
    for _, r in df.iterrows():
        if r['p_good']   >= p_good_thr:                              labels.append('good')
        elif r['p_usable'] >= p_usable_thr:                           labels.append('usable')
        else:                                                          labels.append('reject')
    return labels


def _route_distance(labels):
    cnt = Counter(labels)
    total = sum(cnt.values())
    obs = {k: cnt.get(k, 0) / total for k in TARGET}
    return sum((obs[k] - TARGET[k])**2 for k in TARGET), obs


grid_good   = np.linspace(0.20, 0.80, 31)
grid_usable = np.linspace(0.10, 0.60, 26)
best = (float('inf'), None, None, None)
for tg in grid_good:
    for tu in grid_usable:
        lbls = _route_under(tg, tu, probs_df)
        d, obs = _route_distance(lbls)
        if d < best[0]:
            best = (d, tg, tu, obs)

print(f'Best thresholds: p_good>={best[1]:.3f}, p_usable>={best[2]:.3f}  (sq-dist={best[0]:.4f})')
print(f'Resulting distribution: {best[3]}')

QUALITY_THRESHOLDS = {'good': float(best[1]), 'usable': float(best[2])}
with open(P5_METRICS / 'quality_thresholds.json', 'w') as f:
    json.dump(QUALITY_THRESHOLDS, f, indent=2)

# 3) Replace predict_quality() to use thresholds.
def predict_quality(img):
    """V2 patch: threshold-based routing instead of argmax.

    Returns one of 'good' | 'usable' | 'reject'. Tuned on clean APTOS so the
    routing distribution sits near the 60:25:15 target documented in §1d
    of the master prompt.
    """
    if qmodel is None: return 'good'
    x = TFM_EVAL(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(qmodel(x), 1)[0].cpu().numpy()
    if p[0] >= QUALITY_THRESHOLDS['good']:
        return 'good'
    if p[1] >= QUALITY_THRESHOLDS['usable']:
        return 'usable'
    return 'reject'

# Sanity check on clean APTOS
sanity = Counter(predict_quality(Image.open(resolve_image(sid)).convert('RGB'))
                 for sid in test_ids_q[:100] if Path(resolve_image(sid)).exists())
print(f'Sanity check (100 clean APTOS images): {dict(sanity)}')
print('Phase 5 quality classifier calibrated.')


In [ ]:
# 5.4 Run pipeline on test set: clean + blur-high + exposure-high
EVAL_CONDITIONS = [('clean', None, None),
                    ('blur',     'blur',     'high'),
                    ('exposure', 'exposure', 'high')]

rows = []
for id_code in tqdm(list(test_id_set)[:80], desc='ensemble'):    # cap 80 for time
    if id_code not in labels_df.index: continue
    label = int(labels_df[id_code])
    for tag, kind, level in EVAL_CONDITIONS:
        src = resolve_image(id_code) if tag == 'clean' \
              else find_in_folder(DEGRADED_DIR / kind / level, id_code)
        if not src.exists(): continue
        # Resize the image to IMAGE_SIZE_DOWNSTREAM (384x384) before passing to pipeline
        # This ensures compatibility with the Real-ESRGAN model's internal operations.
        out = pipeline(Image.open(src).convert('RGB').resize((IMAGE_SIZE_DOWNSTREAM, IMAGE_SIZE_DOWNSTREAM)),
                       kind_hint=kind, level_hint=level, id_hint=id_code)
        rows.append({'id_code': id_code, 'condition': tag, 'true_label': label,
                     'pred': out['pred'], 'correct': int(out['pred'] == label),
                     'quality_pred': out['quality_pred'],
                     'route_model': out['route']['model'],
                     'route_enhancement': out['route']['enhancement'],
                     'route_xai': out['route']['xai'],
                     'conf': out['conf'], 'insertion_auc': out['insertion_auc'],
                     'trust': out['trust']})

ens_df = pd.DataFrame(rows)
ens_df.to_csv(P5 / 'metrics' / 'ensemble_predictions.csv', index=False)
ens_df.head()

In [ ]:
# 5.5 Aggregate vs single-best baseline (Phase 2)
summary = (ens_df.groupby('condition')
                  .agg(accuracy=('correct', 'mean'),
                       mean_trust=('trust', 'mean'),
                       mean_insertion=('insertion_auc', 'mean'),
                       n=('id_code', 'count'))).round(4).reset_index()
baseline = (stress_df[stress_df['model'] == best_clean]
             .assign(condition=lambda d: np.where(d['degradation'] == 'clean', 'clean',
                                                  d['degradation'] + '-' + d['level']))
             [['condition', 'accuracy', 'f1_macro']]
             .rename(columns={'accuracy': 'baseline_acc', 'f1_macro': 'baseline_f1'}))
summary = summary.merge(baseline, on='condition', how='left')
summary.to_csv(P5 / 'metrics' / 'ensemble_summary.csv', index=False)
summary

In [ ]:
# 5.6 Plots
fig, ax = plt.subplots(figsize=(7, 4))
for cond in ens_df['condition'].unique():
    ax.hist(ens_df.loc[ens_df['condition'] == cond, 'trust'], bins=15, alpha=0.5, label=cond)
ax.set_title('Clinical Trust Score distribution'); ax.set_xlabel('trust'); ax.legend()
plt.tight_layout()
plt.savefig(P5 / 'plots' / 'trust_score_distribution.png', dpi=150); plt.show()

clean = ens_df[ens_df['condition'] == 'clean']
if not clean.empty:
    cm = confusion_matrix(clean['true_label'], clean['pred'], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title('Confusion matrix — ensemble (clean)'); fig.colorbar(im)
    plt.tight_layout()
    plt.savefig(P5 / 'plots' / 'confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# 5.7 Qualitative — one routed example
demo = ens_df.iloc[0]
src = resolve_image(demo['id_code']) if demo['condition'] == 'clean' \
      else find_in_folder(DEGRADED_DIR / demo['condition'] / 'high', demo['id_code'])
img = Image.open(src).convert('RGB')
out = pipeline(img,
               kind_hint=None if demo['condition'] == 'clean' else demo['condition'],
               level_hint=None if demo['condition'] == 'clean' else 'high',
               id_hint=demo['id_code'])
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(img.resize((IMAGE_SIZE, IMAGE_SIZE))); axes[0].set_title(f'input ({demo["condition"]})')
axes[1].imshow(out['enhanced_img'].resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[1].set_title(f"after {out['route']['enhancement']}")
axes[2].imshow(out['enhanced_img'].resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[2].imshow(out['heatmap'], cmap='jet', alpha=0.45)
axes[2].set_title(f"{out['route']['xai']} — pred={out['pred']} true={demo['true_label']}")
for ax in axes: ax.axis('off')
fig.suptitle(f"Q={out['quality_pred']} | M={out['route']['model']} | trust={out['trust']:.2f}")
plt.tight_layout()
save = P5 / 'samples' / f"route_{demo['id_code']}_{demo['condition']}.png"
plt.savefig(save, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', save)

In [ ]:
import shutil
# Save degraded + enhanced if not already on Drive
for name, src in [('degraded', DEGRADED_DIR), ('enhanced', ENHANCED_DIR)]:
    dest = Path(f'/content/drive/MyDrive/Thesis/{name}')
    if src.exists() and not dest.exists():
        shutil.copytree(src, dest)
        print(f'Saved {name} to Drive')
    elif dest.exists():
        print(f'{name} already on Drive')
    else:
        print(f'{name} not found locally')

In [ ]:
# How often does the quality classifier say 'reject' on test images we know are clean?
from collections import Counter
test_ids = pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].astype(str).tolist()
quality_calls = []
for sid in test_ids[:100]:
    try:
        img = Image.open(resolve_image(sid)).convert('RGB')
        q = predict_quality(img)
        quality_calls.append(q)
    except FileNotFoundError:
        continue
print(Counter(quality_calls))
# Expected on clean APTOS images: mostly 'good', some 'usable', very few 'reject'
# If you see >20% 'reject' here, the classifier is broken

In [ ]:
def fundus_mask(img_pil, size=384):
    arr = np.array(img_pil.resize((size, size)).convert('L'))
    return (arr > 7).astype(np.float32)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# ---- 1. Sample size and condition mix --------------------------------------
N_PER_CONDITION = 25                       # bump if you have time; 25 is enough for variety
TEST_IDS = pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].astype(str).tolist()
SAMPLE_IDS = TEST_IDS[:N_PER_CONDITION]

# Mix of clean + degraded conditions (skip 'high' levels — too easy to fail trivially)
PROBE_CONDITIONS = [
    ('clean', None, None),
    ('noise',    'noise',    'mid'),
    ('blur',     'blur',     'mid'),
    ('exposure', 'exposure', 'mid'),
]


def run_pipeline_and_collect(id_code, tag, kind, level):
    """Run the pipeline on one image, return a dict with everything we need."""
    if tag == 'clean':
        src = resolve_image(id_code)
    else:
        src = DEGRADED_DIR / kind / level / f'{id_code}{DEG_SAVE_EXT}'
    if not src.exists():
        return None
    img = Image.open(src).convert('RGB')
    out = pipeline(img.resize((IMAGE_SIZE_DOWNSTREAM, IMAGE_SIZE_DOWNSTREAM)),
                    kind_hint=kind if tag != 'clean' else None,
                    level_hint=level if tag != 'clean' else None,
                    id_hint=id_code)
    return {
        'id_code': id_code, 'condition': tag,
        'src_img': img,
        'true_label': int(labels_df[id_code]),
        **{k: out[k] for k in ('quality_pred', 'route', 'pred', 'conf',
                                'insertion_auc', 'trust', 'heatmap', 'enhanced_img')},
    }


print(f'Collecting pipeline results: {len(SAMPLE_IDS)} ids × {len(PROBE_CONDITIONS)} conditions ...')
records = []
for sid in tqdm(SAMPLE_IDS, desc='pipeline'):
    if sid not in labels_df.index:
        continue
    for tag, kind, level in PROBE_CONDITIONS:
        r = run_pipeline_and_collect(sid, tag, kind, level)
        if r is not None:
            records.append(r)

print(f'Collected {len(records)} results')


# ---- 2. Classify each record into one of 5 scenarios ----------------------
def classify(rec):
    """Return one of: 'correct_routing', 'restoration_save',
       'mis_routing_lucky', 'pipeline_failure', 'low_trust_correct'."""
    correct = (rec['pred'] == rec['true_label'])
    q = rec['quality_pred']
    cond = rec['condition']

    # Failure: prediction wrong and quality classifier mis-routed
    if not correct and ((cond == 'clean' and q != 'good') or
                         (cond != 'clean' and q == 'good')):
        return 'pipeline_failure'

    # Restoration save: condition is degraded AND quality flagged it correctly
    # AND model got it right (GenAI did its job)
    if correct and cond != 'clean' and q in ('usable', 'reject'):
        return 'restoration_save'

    # Correct routing: clean image, quality says good, prediction correct
    if correct and cond == 'clean' and q == 'good':
        return 'correct_routing'

    # Mis-routing but lucky: clean image, quality wrongly says reject/usable, still correct
    if correct and cond == 'clean' and q != 'good':
        return 'mis_routing_lucky'

    # Low trust correct: correct prediction but trust < 0.5
    if correct and rec['trust'] < 0.5:
        return 'low_trust_correct'

    return None


# Bucket records by scenario; pick the first match per bucket
SCENARIO_ORDER = [
    'correct_routing',
    'restoration_save',
    'mis_routing_lucky',
    'pipeline_failure',
    'low_trust_correct',
]
SCENARIO_LABELS = {
    'correct_routing':   'Correct routing\n(clean -> no enhance -> correct)',
    'restoration_save':  'Restoration save\n(degraded -> GenAI -> correct)',
    'mis_routing_lucky': 'Mis-routing, lucky\n(clean flagged reject -> still correct)',
    'pipeline_failure':  'Pipeline failure\n(wrong route AND wrong prediction)',
    'low_trust_correct': 'Low-trust correct\n(correct but trust < 0.5)',
}

picks = {}
for rec in records:
    s = classify(rec)
    if s is not None and s not in picks:
        picks[s] = rec

# Sanity print
print('\nScenario coverage:')
for s in SCENARIO_ORDER:
    if s in picks:
        r = picks[s]
        print(f"  [OK]   {s:20s} -> id={r['id_code']} cond={r['condition']} "
              f"q={r['quality_pred']} pred={r['pred']} true={r['true_label']} "
              f"trust={r['trust']:.2f}")
    else:
        print(f"  [miss] {s:20s} -> no example found in sample (try larger N_PER_CONDITION)")


# ---- 3. Render 5-row figure -----------------------------------------------
def fundus_mask(img_pil, size):
    arr = np.array(img_pil.resize((size, size)).convert('L'))
    return (arr > 7).astype(np.float32)


def render_panel(ax_row, rec, vis_size=384):
    img      = rec['src_img']
    enhanced = rec['enhanced_img']
    heat     = rec['heatmap']
    mask     = fundus_mask(enhanced, vis_size)
    heat_r   = cv2.resize(heat, (vis_size, vis_size), interpolation=cv2.INTER_LINEAR) * mask

    # col 0 -- input
    ax_row[0].imshow(img.resize((vis_size, vis_size)))
    ax_row[0].set_title(f"input ({rec['condition']})", fontsize=10)
    ax_row[0].axis('off')

    # col 1 -- after enhancement
    ax_row[1].imshow(enhanced.resize((vis_size, vis_size)))
    enh_label = rec['route']['enhancement']
    ax_row[1].set_title(f"after {enh_label}", fontsize=10)
    ax_row[1].axis('off')

    # col 2 -- heatmap overlay
    arr   = np.array(enhanced.resize((vis_size, vis_size))).astype(np.float32) / 255
    over  = 0.55 * arr + 0.45 * plt.get_cmap('jet')(heat_r)[..., :3]
    over  = np.clip(over, 0, 1)
    ax_row[2].imshow(over)
    correct_mark = 'OK' if rec['pred'] == rec['true_label'] else 'X'
    ax_row[2].set_title(
        f"{rec['route']['xai']} - pred={rec['pred']} true={rec['true_label']} [{correct_mark}]",
        fontsize=9)
    ax_row[2].axis('off')


present = [s for s in SCENARIO_ORDER if s in picks]
if not present:
    print('\nNo scenarios matched — nothing to render.')
else:
    fig, axes = plt.subplots(len(present), 3, figsize=(12, 3.5 * len(present)))
    if len(present) == 1:
        axes = axes[None, :]
    for i, s in enumerate(present):
        rec = picks[s]
        render_panel(axes[i], rec)
        # Row label as text on the left
        axes[i, 0].text(-0.15, 0.5,
                         SCENARIO_LABELS[s],
                         transform=axes[i, 0].transAxes,
                         fontsize=11, fontweight='bold', va='center', ha='right',
                         rotation=0)
        # Per-row summary subtitle above row
        rec_meta = (f"Q={rec['quality_pred']} | M={rec['route']['model']} | "
                    f"enh={rec['route']['enhancement']} | "
                    f"trust={rec['trust']:.2f} | conf={rec['conf']:.2f}")
        axes[i, 1].set_xlabel(rec_meta, fontsize=8)

    fig.suptitle('Pipeline behaviour across 5 scenarios', fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0.05, 0, 1, 0.97])
    out = P5 / 'samples' / 'five_scenarios.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('\nSaved:', out)

# ---- 4. Save the picks as a CSV for the dissertation appendix --------------
if picks:
    picks_csv = P5 / 'metrics' / 'five_scenarios.csv'
    pd.DataFrame([
        {'scenario': s,
         'id_code': picks[s]['id_code'],
         'condition': picks[s]['condition'],
         'quality_pred': picks[s]['quality_pred'],
         'route_model': picks[s]['route']['model'],
         'route_enhancement': picks[s]['route']['enhancement'],
         'pred': picks[s]['pred'],
         'true': picks[s]['true_label'],
         'conf': round(picks[s]['conf'], 3),
         'insertion_auc': round(picks[s]['insertion_auc'], 3),
         'trust': round(picks[s]['trust'], 3)}
        for s in SCENARIO_ORDER if s in picks
    ]).to_csv(picks_csv, index=False)
    print('Picks CSV:', picks_csv)

In [ ]:
# === Severe DR + blur progression — before/after GenAI explanation ===
# For one Severe-DR (class 3) test image, runs the model + XAI on:
#   - clean reference
#   - blur low/mid/high (raw)
#   - blur low/mid/high (GenAI-restored)
# Renders one 4x4 figure per model + saves an all-metrics CSV.

TARGET_CLASS = 3                                   # 3 = Severe DR
ALL_LEVELS   = ['low', 'mid', 'high']

# ---- 1. Find a Severe-DR test image the strongest v3 model is confident on
test_ids_set = set(pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].astype(str))
severe_ids   = labels_df[labels_df == TARGET_CLASS].index.astype(str).tolist()
severe_test  = [i for i in severe_ids if i in test_ids_set]
print(f'Severe DR images in test set: {len(severe_test)}')

# Score each candidate by softmax-confidence-for-class-3 on the clean image
ref_model_name = MODEL_NAMES[0]
ref_model      = models[ref_model_name]

def _clean_conf(id_code):
    x = load_tensor(resolve_image(id_code))
    with torch.no_grad():
        return float(torch.softmax(ref_model(x), 1)[0, TARGET_CLASS].item())

scored  = sorted(severe_test, key=_clean_conf, reverse=True)
demo_id = scored[0]
print(f'Picked demo_id={demo_id} (clean confidence for Severe = {_clean_conf(demo_id):.3f})')


# ---- 2. Helpers: heatmap function per model + metric collection
def _get_fn(name):
    if name == 'vit_base':
        return attention_rollout, 'attention'
    return gradcam_heatmap, 'gradcam'


def _metrics(model, x, heat, h_clean=None):
    with torch.no_grad():
        prob = torch.softmax(model(x), 1)[0]
    return {
        'pred':           int(prob.argmax().item()),
        'conf_severe':    float(prob[TARGET_CLASS].item()),
        'insertion_auc':  round(insertion_auc(model, x, heat, TARGET_CLASS), 4),
        'deletion_auc':   round(deletion_auc(model, x, heat, TARGET_CLASS), 4),
        'stability':      1.0 if h_clean is None else round(stability_spearman(h_clean, heat), 4),
    }


# ---- 3. Run pipeline: for each (model, level), compute raw + enhanced
results = []
panels  = {}             # panels[(name, level, variant)] = (img, heat, metrics)

img_clean = Image.open(resolve_image(demo_id)).convert('RGB')
x_clean   = load_tensor(resolve_image(demo_id))

for name in MODEL_NAMES:
    model    = models[name]
    fn, method_name = _get_fn(name)
    print(f'\n--- {name} / {method_name} ---')

    # Clean reference
    h_clean = fn(model, x_clean, target_class=TARGET_CLASS)
    m = _metrics(model, x_clean, h_clean)
    panels[(name, 'clean', 'reference')] = (img_clean, h_clean, m)
    results.append({'model': name, 'method': method_name, 'level': 'clean',
                    'variant': 'reference', **m})
    print(f"  clean: pred={m['pred']}  conf_severe={m['conf_severe']:.3f}")

    for level in ALL_LEVELS:
        # ---- raw degraded ----
        raw_path = DEGRADED_DIR / 'blur' / level / f'{demo_id}{DEG_SAVE_EXT}'
        if not raw_path.exists():
            print(f"  [skip] missing {raw_path}")
            continue
        raw_img = Image.open(raw_path).convert('RGB')
        x_raw   = load_tensor(raw_path)
        h_raw   = fn(model, x_raw, target_class=TARGET_CLASS)
        m_raw   = _metrics(model, x_raw, h_raw, h_clean)
        panels[(name, level, 'raw')] = (raw_img, h_raw, m_raw)
        results.append({'model': name, 'method': method_name, 'level': level,
                        'variant': 'raw', **m_raw})

        # ---- GenAI-restored ----
        enh_img = enhance_genai(raw_img)
        # Pipe through the same eval transform the model expects
        x_enh   = TFM_EVAL(enh_img).unsqueeze(0).to(DEVICE)
        h_enh   = fn(model, x_enh, target_class=TARGET_CLASS)
        m_enh   = _metrics(model, x_enh, h_enh, h_clean)
        panels[(name, level, 'genai')] = (enh_img, h_enh, m_enh)
        results.append({'model': name, 'method': method_name, 'level': level,
                        'variant': 'genai', **m_enh})

        print(f"  blur-{level}  raw : pred={m_raw['pred']} conf={m_raw['conf_severe']:.3f} "
              f"ins={m_raw['insertion_auc']:.3f} del={m_raw['deletion_auc']:.3f} stab={m_raw['stability']:.3f}")
        print(f"  blur-{level}  genai: pred={m_enh['pred']} conf={m_enh['conf_severe']:.3f} "
              f"ins={m_enh['insertion_auc']:.3f} del={m_enh['deletion_auc']:.3f} stab={m_enh['stability']:.3f}")


# ---- 4. Save the full metric table
results_df = pd.DataFrame(results)
out_csv = P4 / 'metrics' / f'severe_blur_progression_{demo_id}.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_csv, index=False)
print(f'\nSaved metrics: {out_csv}')
print(results_df.to_string(index=False))


# ---- 5. Render one 4x4 figure per model (clean ref + 3 blur levels)
VIS_SIZE = 384


def fundus_mask(img_pil, size):
    arr = np.array(img_pil.resize((size, size)).convert('L'))
    return (arr > 7).astype(np.float32)


def _ov(img, heat, size=VIS_SIZE):
    mask     = fundus_mask(img, size)
    arr      = np.array(img.resize((size, size))).astype(np.float32) / 255
    heat_r   = cv2.resize(heat, (size, size), interpolation=cv2.INTER_LINEAR) * mask
    return np.clip(0.55 * arr + 0.45 * plt.get_cmap('jet')(heat_r)[..., :3], 0, 1)


def _title(m, level, variant):
    ok = 'OK' if m['pred'] == TARGET_CLASS else 'X'
    return (f"{variant} {level} [{ok}]\n"
            f"pred={m['pred']}  conf={m['conf_severe']:.2f}\n"
            f"Ins:{m['insertion_auc']:.2f}  Del:{m['deletion_auc']:.2f}  Stab:{m['stability']:.2f}")


figs_saved = []
for name in MODEL_NAMES:
    fn, method_name = _get_fn(name)
    fig, axes = plt.subplots(4, 4, figsize=(15, 14))

    # Row 0 — clean reference (cols 2-3 left blank with 'reference')
    img, heat, m = panels[(name, 'clean', 'reference')]
    axes[0, 0].imshow(img.resize((VIS_SIZE, VIS_SIZE)))
    axes[0, 0].set_title('CLEAN input', fontsize=10)
    axes[0, 1].imshow(_ov(img, heat))
    axes[0, 1].set_title(f"CLEAN {method_name}\npred={m['pred']} conf={m['conf_severe']:.2f}", fontsize=9)
    for c in (2, 3):
        axes[0, c].text(0.5, 0.5, '(no GenAI on clean)',
                         transform=axes[0, c].transAxes, ha='center', va='center', fontsize=11,
                         color='gray', style='italic')
        axes[0, c].set_facecolor('#f5f5f5')

    # Rows 1-3 — blur levels (raw + GenAI side-by-side)
    for r, level in enumerate(ALL_LEVELS, start=1):
        # Raw
        if (name, level, 'raw') in panels:
            img, heat, m = panels[(name, level, 'raw')]
            axes[r, 0].imshow(img.resize((VIS_SIZE, VIS_SIZE)))
            axes[r, 0].set_title(f'blur-{level}  RAW', fontsize=10)
            axes[r, 1].imshow(_ov(img, heat))
            axes[r, 1].set_title(_title(m, level, 'raw'), fontsize=8)
        # GenAI
        if (name, level, 'genai') in panels:
            img, heat, m = panels[(name, level, 'genai')]
            axes[r, 2].imshow(img.resize((VIS_SIZE, VIS_SIZE)))
            axes[r, 2].set_title(f'blur-{level}  GenAI', fontsize=10)
            axes[r, 3].imshow(_ov(img, heat))
            axes[r, 3].set_title(_title(m, level, 'genai'), fontsize=8)

    for ax in axes.flat:
        ax.axis('off')

    fig.suptitle(f"{name} -- Severe DR (class {TARGET_CLASS}) | id={demo_id} | "
                  f"blur progression, before vs after GenAI",
                  fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    out = P4 / 'samples' / f'severe_blur_genai_{name}_{demo_id}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    figs_saved.append(out)
    print(f'Saved: {out}')

print(f'\n{len(figs_saved)} model figures saved to {P4 / "samples"}')

## V6 - Low+mid restoration scoping + Phase 5 policy fix

Final scoping pass based on the observation that high-severity inputs are clinically ungradable: restoration is only attempted at **low** and **mid** severity. The raw line in the Phase 4 plots still spans all three levels so the degradation cliff is visible, but enhancer lines stop at mid. One supplementary figure shows the cold-diffusion noise-high collapse as a documented failure mode. Phase 5 `reject` route is changed from "enhance with GenAI" to "no enhancement, flag for re-acquisition" so the routing policy is consistent with the Phase 4 scoping.


In [ ]:
# ============================================================================
# PASTE THIS AS A NEW CELL AT THE BOTTOM OF YOUR RUNNING COLAB NOTEBOOK.
# It uses what's already in memory (rec_df, xai_rec, enhance_cold_diffusion,
# the loaded models, the cached JPEGs on Drive). No re-upload, no re-run
# of Phase 1-3 needed.
#
# What it does:
#   1. Defines RESTORE_LEVELS = ('low', 'mid')
#   2. Re-renders Phase 4 accuracy plots: raw line spans all 3 levels,
#      enhancer lines stop at mid (visual scoping statement)
#   3. Re-renders Phase 4 XAI plots with the same scoping
#   4. Builds ONE supplementary figure: "Why we don't restore at high" —
#      a noise-high cold_diff failure case
#   5. Patches QUALITY_POLICY['reject'] -> no enhancement + flag for
#      re-acquisition, and re-runs the Phase 5 routed pipeline
# ============================================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

# ---- 1. Scope definition ----
RESTORE_LEVELS = ('low', 'mid')                              # NEW: restoration scope
ALL_LEVELS     = ('low', 'mid', 'high')                       # raw still shown at high

P4_PLOTS_V2 = P4 / 'plots' / 'low_mid_scope'
P4_SAMPS_V2 = P4 / 'samples' / 'low_mid_scope'
P4_PLOTS_V2.mkdir(parents=True, exist_ok=True)
P4_SAMPS_V2.mkdir(parents=True, exist_ok=True)

# ---- 2. Re-render Phase 4 accuracy plots with scoped enhancer lines ----
print("=== Phase 4 accuracy plots (low+mid restoration scope) ===")
for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(1, len(MODEL_NAMES),
                             figsize=(4.4*len(MODEL_NAMES), 4), sharey=True)
    for ax, name in zip(axes, MODEL_NAMES):
        sub = rec_df[(rec_df['degradation'] == kind) & (rec_df['model'] == name)].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=list(ALL_LEVELS), ordered=True)
        for variant in VARIANTS:
            d = sub[sub['variant'] == variant].sort_values('level')
            if variant == 'raw':
                ax.plot(d['level'], d['accuracy'], marker='o', linewidth=2.2,
                        label='raw (no restoration)', color='black')
            else:
                d_scope = d[d['level'].isin(RESTORE_LEVELS)]
                ax.plot(d_scope['level'], d_scope['accuracy'],
                        marker='o', linestyle='--', label=variant)
        ax.axvspan(2 - 0.5, 2 + 0.5, color='lightcoral', alpha=0.15)   # shade high
        ax.text(2, 0.05, 'ungradable\n(no restoration)',
                ha='center', va='bottom', fontsize=8, style='italic', color='darkred')
        ax.set_title(name); ax.set_xlabel('severity'); ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel('accuracy'); axes[-1].legend(loc='lower left', fontsize=8)
    fig.suptitle(f'Accuracy under {kind} — restoration scoped to low+mid')
    plt.tight_layout()
    out = P4_PLOTS_V2 / f'scoped_accuracy_{kind}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f"  Saved: {out}")

# ---- 3. Re-render XAI plots with same scoping ----
print("\n=== Phase 4 XAI plots (low+mid restoration scope) ===")
for metric in ('stability', 'insertion_auc'):
    for kind in DEGRADATION_TYPES:
        sub = xai_rec[xai_rec['degradation'] == kind].copy()
        if sub.empty:
            continue
        sub['level'] = pd.Categorical(sub['level'], categories=list(ALL_LEVELS), ordered=True)
        g = (sub.groupby(['model', 'variant', 'level'], observed=True)[metric]
             .mean().reset_index())
        fig, axes = plt.subplots(1, len(MODEL_NAMES),
                                 figsize=(4.4*len(MODEL_NAMES), 4), sharey=True)
        for ax, name in zip(axes, MODEL_NAMES):
            for variant in VARIANTS:
                d = g[(g['model'] == name) & (g['variant'] == variant)].sort_values('level')
                if variant == 'raw':
                    ax.plot(d['level'], d[metric], marker='o', linewidth=2.2,
                            color='black', label='raw')
                else:
                    d_scope = d[d['level'].isin(RESTORE_LEVELS)]
                    ax.plot(d_scope['level'], d_scope[metric],
                            marker='o', linestyle='--', label=variant)
            ax.axvspan(2 - 0.5, 2 + 0.5, color='lightcoral', alpha=0.15)
            ax.set_title(name); ax.grid(alpha=0.3); ax.set_xlabel('severity')
        axes[0].set_ylabel(metric); axes[-1].legend(loc='best', fontsize=8)
        fig.suptitle(f'XAI {metric} — {kind} (restoration scoped to low+mid)')
        plt.tight_layout()
        out = P4_PLOTS_V2 / f'scoped_xai_{metric}_{kind}.png'
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
        print(f"  Saved: {out}")

# ---- 4. "Why we don't restore at high" supplementary figure ----
print("\n=== Supplementary: restoration failure at noise-high ===")

# Pick a representative test id we already have noise-high data for
demo_id = None
for cand in list(test_id_set)[:25]:
    try:
        _ = find_in_folder(DEGRADED_DIR / 'noise' / 'high', cand)
        _ = find_in_folder(ENHANCED_DIR / 'cold_diff' / 'noise' / 'high', cand)
        demo_id = cand
        break
    except FileNotFoundError:
        continue

if demo_id is None:
    print("  [skip] no test id with both noise-high degraded + cold_diff variant on disk")
else:
    clean_pil = Image.open(resolve_image(demo_id)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    deg_pil   = Image.open(find_in_folder(DEGRADED_DIR / 'noise' / 'high', demo_id)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    panels = [(clean_pil, 'clean (reference)'), (deg_pil, 'noise-high input\n"ungradable"')]
    for variant in ENHANCERS:
        try:
            v_pil = Image.open(find_in_folder(ENHANCED_DIR / variant / 'noise' / 'high', demo_id)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
            # Look up classifier accuracy for this slice from rec_df (averaged across models)
            sub_acc = rec_df[(rec_df['degradation'] == 'noise') &
                             (rec_df['level'] == 'high') &
                             (rec_df['variant'] == variant)]['accuracy'].mean()
            panels.append((v_pil, f'{variant}\nacc={sub_acc:.3f}'))
        except FileNotFoundError:
            panels.append((Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE)),
                           f'{variant}\n(missing)'))

    # Also append the raw baseline accuracy for context
    raw_acc = rec_df[(rec_df['degradation'] == 'noise') &
                     (rec_df['level'] == 'high') &
                     (rec_df['variant'] == 'raw')]['accuracy'].mean()
    panels[1] = (deg_pil, f'noise-high input\nraw acc={raw_acc:.3f}\n(no restoration)')

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3.6))
    for ax, (im, ttl) in zip(axes, panels):
        ax.imshow(im); ax.set_title(ttl, fontsize=10); ax.axis('off')
    fig.suptitle(f'Why we stop at mid — restoration at noise-high (id={demo_id})',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    out = P4_SAMPS_V2 / f'failure_noise_high_{demo_id}.png'
    plt.savefig(out, dpi=160, bbox_inches='tight'); plt.show()
    print(f"  Saved: {out}")
    print(f"\n  Headline numbers from rec_df at noise-high:")
    nh = rec_df[(rec_df['degradation'] == 'noise') & (rec_df['level'] == 'high')]
    print(nh.groupby('variant')['accuracy'].mean().round(3).to_string())

# ---- 5. Patch Phase 5 routing policy ----
print("\n=== Phase 5 routing policy update ===")
try:
    OLD_POLICY = dict(QUALITY_POLICY)
    QUALITY_POLICY['reject'] = {
        'enhancement': 'none',
        'model': 'resnet50',
        'xai': 'gradcam',
        'flag': 'reacquire',          # NEW: explicit re-acquisition flag
    }
    print("  Old reject path:", OLD_POLICY['reject'])
    print("  New reject path:", QUALITY_POLICY['reject'])
    print("\n  Re-run cells 82 (route pipeline) and 83 (aggregate) to see the")
    print("  updated headline numbers with the corrected policy.")
except NameError:
    print("  [skip] QUALITY_POLICY not in memory — run Phase 5 cells first.")

print("\nDone. New artifacts under:")
print(f"  Plots:   {P4_PLOTS_V2}")
print(f"  Samples: {P4_SAMPS_V2}")


## V2 — Step 5 + 6 — Per-class diagnostics + failure-mode analysis

Concluding diagnostics for the dissertation chapter:

1. Per-class precision / recall / F1 for the V4 ensemble on the clean test set and on each
   high-severity degradation slice. Plus a confusion matrix figure per slice.
2. Failure-mode table: per-image `raw_pred` vs `enhanced_pred` grouped by `true_class`, showing
   which classes benefit most from restoration and which (if any) are harmed.


In [ ]:
# === V2 PATCH (Step 5+6): per-class diagnostics + failure-mode analysis ===
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

TARGET_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# Use the V4 ensemble predictions cached during Phase 2 (cell 41).
ens_csv = P2 / 'metrics' / 'v4' / 'v4_ensemble_predictions.csv'
if not ens_csv.exists():
    # Fallback: re-evaluate just the clean test set so this cell is still useful.
    print(f'[note] {ens_csv} not found — running clean V4 ensemble eval inline')
    _tl = globals().get('test_loader_v3') or globals().get('test_loader')
    if _tl is None:
        print('[skip] no test loader in memory — run Phase 2 cells first')
        ens_df_v4 = pd.DataFrame()
    else:
        rows = []
        for batch in tqdm(_tl, desc='V4 clean recompute'):
            if len(batch) == 3:
                x, y, ids = batch
            else:
                x, y = batch[0], batch[1]
                ids = [f'idx_{i}' for i in range(len(y))]
            x = x.to(DEVICE, non_blocking=True)
            probs = None
            for name in MODEL_NAMES:
                mdl = load_classifier_v3(name)
                with torch.no_grad():
                    p = torch.softmax(mdl(x), 1)
                probs = p if probs is None else probs + p
                del mdl; torch.cuda.empty_cache()
            probs = probs / len(MODEL_NAMES)
            pred = probs.argmax(1).cpu().numpy()
            for yi, pi, ci in zip(y.numpy(), pred, ids):
                rows.append({'id_code': ci, 'true_label': int(yi), 'pred': int(pi), 'condition': 'clean'})
        ens_df_v4 = pd.DataFrame(rows)
else:
    ens_df_v4 = pd.read_csv(ens_csv)

print('V4 ensemble predictions:', ens_df_v4.shape)

# --- 1. Per-class classification report (clean) ---
clean = ens_df_v4[ens_df_v4.get('condition', 'clean') == 'clean']
if len(clean) > 0:
    print('\n=== V4 ensemble — clean test set ===')
    print(classification_report(clean['true_label'], clean['pred'],
                                target_names=TARGET_NAMES, zero_division=0))
    cm = confusion_matrix(clean['true_label'], clean['pred'], labels=list(range(5)))
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES, cmap='Blues', ax=ax)
    ax.set_title('V4 Ensemble — Clean Test Set')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    cm_out = P2 / 'plots' / 'v4_confusion_clean.png'
    plt.savefig(cm_out, dpi=160, bbox_inches='tight'); plt.show()
    print(f'Saved: {cm_out}')

# --- 2. Failure-mode analysis: raw vs enhanced per true class (high severity only) ---
if 'rec_df' in globals():
    print('\n=== Failure-mode: per-class accuracy delta with each enhancer (high severity) ===')
    failure_rows = []
    for variant in VARIANTS:
        for k in DEGRADATION_TYPES:
            # Only high severity to keep runtime reasonable.
            l = 'high'
            root = DEGRADED_DIR / k / l if variant == 'raw' else ENHANCED_DIR / variant / k / l
            if not (root / 'manifest.csv').exists():
                continue
            ds = FolderDataset(root, transform=TFM_EVAL)
            ds.df = ds.df[ds.df['id_code'].astype(str).isin(test_id_set)].reset_index(drop=True)
            dl = DataLoader(ds, batch_size=32, num_workers=2, pin_memory=True)
            preds_acc = {n: [] for n in MODEL_NAMES}
            ys_all, ids_all = [], []
            for n in MODEL_NAMES:
                mdl = load_classifier_v3(n) if 'load_classifier_v3' in globals() else load_classifier(n)
                for x, y, ids in dl:
                    with torch.no_grad():
                        p = torch.softmax(mdl(x.to(DEVICE)), 1)
                    preds_acc[n].append(p.cpu().numpy())
                    if n == MODEL_NAMES[0]:
                        ys_all.append(y.numpy()); ids_all.extend(ids)
                del mdl; torch.cuda.empty_cache()
            ys_all = np.concatenate(ys_all)
            avgp   = sum(np.concatenate(preds_acc[n]) for n in MODEL_NAMES) / len(MODEL_NAMES)
            pred   = avgp.argmax(1)
            for ci, yi, pi in zip(ids_all, ys_all, pred):
                failure_rows.append({'id_code': ci, 'true_label': int(yi),
                                     'pred': int(pi), 'variant': variant,
                                     'degradation': k, 'level': l,
                                     'correct': int(pi == yi)})
    failure_df = pd.DataFrame(failure_rows)
    failure_df.to_csv(P4 / 'metrics' / 'failure_mode_table.csv', index=False)
    print('\nPer-(true_label, variant) accuracy at high severity (avg across kinds):')
    print(failure_df.groupby(['true_label', 'variant'])['correct'].mean().unstack().round(3).to_string())
    print('\nPer-(degradation, variant) accuracy:')
    print(failure_df.groupby(['degradation', 'variant'])['correct'].mean().unstack().round(3).to_string())
else:
    print('[skip] rec_df not in memory — run cell 72 first')


## V3 — Step 8 — Hallucination detection (pathology DDPM)

A restoration *hallucinates* when it changes the diagnostic content rather than
just the image quality. The clearest operational signal is a **change in the
predicted DR grade** between the degraded input and the restored output (a
lesion added or erased). For a capped sample of high-severity test images we
log, per degradation kind:

- DR grade + confidence before/after restoration, and the grade-change rate
- mean pixel deviation (L1 in [0,1]) between restored and degraded
- a 0–3 risk score → low / medium / high

Saved as a CSV under the Phase-4 metrics folder (no new plots).


In [ ]:
# === V3 PATCH (Step 8): hallucination detection for the pathology DDPM ===
import numpy as np
import torch.nn.functional as F

P4 = PHASE_DIRS['phase4_genai_enhancement']
(P4 / 'metrics').mkdir(parents=True, exist_ok=True)

_hall_clf = load_classifier_v3('efficientnet_b3')   # our trained DR grader
_hall_clf.eval()


def _clf_logits(pil):
    x = TFM_EVAL_V3(pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return _hall_clf(x)


def hallucination_check(degraded_pil, restored_pil):
    lb = _clf_logits(degraded_pil); la = _clf_logits(restored_pil)
    gb, ga = int(lb.argmax(1)), int(la.argmax(1))
    cb = float(F.softmax(lb, 1).max()); ca = float(F.softmax(la, 1).max())
    db = np.asarray(degraded_pil.resize((256, 256)), dtype=np.float32) / 255.0
    dr = np.asarray(restored_pil.resize((256, 256)), dtype=np.float32) / 255.0
    pix = float(np.abs(dr - db).mean())
    score = (2 if gb != ga else 0) + (1 if pix > 0.15 else 0)
    risk = 'low' if score <= 1 else ('medium' if score == 2 else 'high')
    return dict(grade_before=gb, grade_after=ga, grade_changed=gb != ga,
                conf_before=round(cb, 3), conf_after=round(ca, 3),
                pixel_deviation=round(pix, 3),
                risk_score=score, hallucination_risk=risk)


# Run over a capped sample of high-severity test images (time budget).
_hall_rows = []
_sample = set(map(str, list(test_id_set)[:60]))
for k in DEGRADATION_TYPES:
    l = 'high'
    src_dir = DEGRADED_DIR / k / l
    if not (src_dir / 'manifest.csv').exists():
        continue
    mani = pd.read_csv(src_dir / 'manifest.csv')
    mani = mani[mani['id_code'].astype(str).isin(_sample)]
    for _, row in tqdm(mani.iterrows(), total=len(mani),
                       desc=f'hallucination {k}/{l}', leave=False):
        deg = Image.open(src_dir / row['rel_path']).convert('RGB')
        res = enhance_ddpm_path(deg)
        r = hallucination_check(deg, res)
        r.update(id_code=row['id_code'], kind=k, level=l)
        _hall_rows.append(r)

hall_df = pd.DataFrame(_hall_rows)
_hall_csv = P4 / 'metrics' / 'ddpm_pathology_hallucination.csv'
hall_df.to_csv(_hall_csv, index=False)
print(f"Saved hallucination report -> {_hall_csv}")
if len(hall_df):
    print(f"  grade-change (hallucination) rate: {hall_df['grade_changed'].mean()*100:.1f}%")
    print("  risk distribution:")
    print(hall_df['hallucination_risk'].value_counts().to_string())
